# Pipeline completo PyCaret — Classificação por Quadrantes Emocionais (Valence–Arousal)

Este notebook cria um pipeline completo para transformar `valence` e `arousal` em **quadrantes emocionais** e treinar modelos de **classificação multiclasse** com features openSMILE.

Fluxo implementado:

1. leitura das features openSMILE e dos rótulos dinâmicos de `valence`/`arousal`;
2. alinhamento temporal por `song_id`;
3. criação dos quatro quadrantes emocionais;
4. agregação opcional por blocos de 10s;
5. análise exploratória, inventário e qualidade das features;
6. validação oficial com `GroupKFold` por `song_id`;
7. benchmark complementar com PyCaret;
8. exportação de tabelas, gráficos, modelo final e resumo para TCC.

Quadrantes usados:

| Quadrante | Condição | Interpretação |
|---|---|---|
| Q1 | valence alta + arousal alto | Alegre / Energético |
| Q2 | valence baixa + arousal alto | Tenso / Raivoso |
| Q3 | valence baixa + arousal baixo | Triste / Melancólico |
| Q4 | valence alta + arousal baixo | Calmo / Relaxado |

**Regra metodológica importante:** a validação oficial usa `GroupKFold` por `song_id`, para evitar que blocos/frames da mesma música apareçam simultaneamente no treino e no teste.

> `song_id` é usado apenas como grupo de validação e nunca como feature preditora. A métrica principal para discussão é `macro-F1` da validação oficial com `GroupKFold`.


## Declaração de Baseline — openSMILE como referência metodológica

> **Este notebook avalia exclusivamente o baseline baseado nas features openSMILE fornecidas pela DEAM.** O objetivo é estabelecer uma referência de desempenho para a classificação dos quadrantes emocionais no espaço Valence–Arousal. Os resultados obtidos serão posteriormente comparados com as representações baseadas em fingerprinting emocional.

**Contexto metodológico do TCC:** O openSMILE é amplamente utilizado como extrator de features acústicas de baixo e médio nível (timbre, dinâmica, ritmo, pitch, expressividade). Neste trabalho, ele representa a abordagem **baseline com descritores tradicionais**, enquanto a proposta principal de **fingerprinting emocional** constitui a contribuição central do TCC.

**O que este notebook entrega:**

| Item | Status |
|---|---|
| GroupKFold por `song_id` (sem vazamento) | ✅ |
| Comparação com DummyClassifier (validade do aprendizado) | ✅ |
| Tabela fold a fold com classe mais difícil | ✅ |
| GroupKFold vs StratifiedGroupKFold | ✅ |
| Comparação de seleção de features (VarianceThreshold, SelectKBest) | ✅ |
| Análise da contribuição por categoria acústica | ✅ |
| Matriz de confusão normalizada por linha (%) | ✅ |
| Auditoria anti-vazamento documentada | ✅ |
| Texto interpretativo pronto para o TCC | ✅ |

## 0. Instalação e imports

Execute esta célula apenas se o ambiente ainda não tiver os pacotes necessários. Se você já estiver com o ambiente configurado, pode comentar as chamadas `pip_install`.


In [1]:
import sys
import subprocess


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *packages])


print(f"Python atual: {sys.version}")

INSTALL_PACKAGES = False  # altere para True se estiver em ambiente limpo

if INSTALL_PACKAGES:
    if sys.version_info[:2] >= (3, 12):
        base_packages = [
            "numpy", "pandas>=2.0", "scipy", "scikit-learn", "plotly",
            "nbformat", "tqdm", "openpyxl", "kaleido"
        ]
    else:
        base_packages = [
            "numpy<2.0", "pandas<2.2.0", "scipy<=1.11.4", "scikit-learn",
            "plotly", "nbformat", "tqdm", "openpyxl", "kaleido"
        ]
    pip_install(*base_packages)
    # Para PyCaret 3.x em Python <= 3.11:
    # pip_install("pycaret[classification]")
    # Para Python 3.12+, use sua instalação já validada do PyCaret 4.0, se aplicável.


Python atual: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


In [2]:
import os
import re
import glob
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import GroupKFold, KFold, cross_validate
try:
    from sklearn.model_selection import StratifiedGroupKFold
except Exception:
    StratifiedGroupKFold = None
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    make_scorer,
    ConfusionMatrixDisplay,
)

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# ── PyCaret Classification ──────────────────────────────────────────────────
PYCARET_AVAILABLE = False
PYCARET_IMPORT_ERROR = None
try:
    from pycaret.classification import (
        setup,
        compare_models,
        tune_model,
        finalize_model,
        save_model,
        pull,
        predict_model,
    )
    PYCARET_AVAILABLE = True
except Exception as e:
    PYCARET_IMPORT_ERROR = e
    print("[AVISO] PyCaret Classification não disponível neste ambiente.")
    print(e)

print("PyCaret disponível:", PYCARET_AVAILABLE)


PyCaret disponível: True


## 1. Configuração

Ajuste os caminhos para o seu ambiente. O notebook aceita dois modos:

1. carregar diretamente `features`, `valence.csv` e `arousal.csv`;
2. carregar um dataset já alinhado, se você já tiver salvo um CSV/parquet com features + `valence` + `arousal`.


In [3]:
# =========================
# CAMINHOS
# =========================
# Cache openSMILE por bloco — gerado pelo notebook 00_construir_opensmile_blocks
OPENSMILE_BLOCKS_PATH = r"C:\dev\python\TCC Ajustado\Dados\comparative_outputs\common\opensmile_blocks.parquet"

OUTPUT_PATH = r"C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

FIGURES_PATH = os.path.join(OUTPUT_PATH, "figures")
TABLES_PATH  = os.path.join(OUTPUT_PATH, "tables")
MODELS_PATH  = os.path.join(OUTPUT_PATH, "models")
for p in [FIGURES_PATH, TABLES_PATH, MODELS_PATH]:
    os.makedirs(p, exist_ok=True)

# =========================
# COLUNAS (nomes no parquet opensmile_blocks)
# =========================
SONG_ID_COL  = "song_id"
BLOCK_ID_COL = "block_id"
VALENCE_COL  = "valence"
AROUSAL_COL  = "arousal"
QUADRANT_COL = "quadrant_label"
TIME_COL     = "frametime"   # não existe no parquet — mantido para auditoria de vazamento

# Prefixos de feature (definidos no notebook 00_construir_opensmile_blocks)
MEAN_PREFIX    = "os_mean__"  # média dos frames no bloco
STD_PREFIX     = "os_std__"   # desvio padrão dos frames no bloco
FEATURE_PREFIX = "os_"        # prefixo comum para detecção de todas as features

# =========================
# QUADRANTES
# Labels gerados por compute_quadrant em 00_construir_opensmile_blocks (threshold=0, escala -1..1)
# =========================
QUADRANT_ORDER = [
    "Q1_High_Valence_High_Arousal",
    "Q2_Low_Valence_High_Arousal",
    "Q3_Low_Valence_Low_Arousal",
    "Q4_High_Valence_Low_Arousal",
]

QUADRANT_DESCRIPTIONS = {
    "Q1_High_Valence_High_Arousal": "valence alta + arousal alto",
    "Q2_Low_Valence_High_Arousal":  "valence baixa + arousal alto",
    "Q3_Low_Valence_Low_Arousal":   "valence baixa + arousal baixo",
    "Q4_High_Valence_Low_Arousal":  "valence alta + arousal baixo",
}

QUADRANT_COLOR_MAP = {
    "Q1_High_Valence_High_Arousal": "#2E7D32",
    "Q2_Low_Valence_High_Arousal":  "#C62828",
    "Q3_Low_Valence_Low_Arousal":   "#1565C0",
    "Q4_High_Valence_Low_Arousal":  "#F9A825",
}

QUADRANT_SHORT_LABEL = {
    "Q1_High_Valence_High_Arousal": "Q1 Alegre/Energético",
    "Q2_Low_Valence_High_Arousal":  "Q2 Tenso/Raivoso",
    "Q3_Low_Valence_Low_Arousal":   "Q3 Triste/Melancólico",
    "Q4_High_Valence_Low_Arousal":  "Q4 Calmo/Relaxado",
}

# =========================
# MODELAGEM
# =========================
RANDOM_STATE = 42
N_SPLITS = 5
SELECTOR_K_VALUES = [30, 60, 100, 200]
MAX_MISSING_RATE = 0.40
MAX_SAMPLE_FOR_EDA = 50000
RUN_PYCARET = True
RUN_TUNING = False

PYCARET_FIX_IMBALANCE = True

print("OUTPUT_PATH :", OUTPUT_PATH)
print("Parquet     :", OPENSMILE_BLOCKS_PATH)
print("Protocolo   : GroupKFold por song_id — 1 linha por bloco de 10s")
print(f"Prefíxos    : mean={MEAN_PREFIX!r}  std={STD_PREFIX!r}")

# =========================
# CONFIGURAÇÃO VISUAL
# =========================
EXPORT_PNG = True
EXPORT_SVG = True
FIG_WIDTH  = 1200
FIG_HEIGHT = 720
FIG_SCALE  = 2
PLOT_TEMPLATE = "plotly_white"
SHOW_FIGURES  = True


OUTPUT_PATH : C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs
Parquet     : C:\dev\python\TCC Ajustado\Dados\comparative_outputs\common\opensmile_blocks.parquet
Protocolo   : GroupKFold por song_id — 1 linha por bloco de 10s
Prefíxos    : mean='os_mean__'  std='os_std__'


## 2. Funções utilitárias de leitura, exportação e preparação


In [4]:
def save_table(df_out, filename):
    path = os.path.join(TABLES_PATH, filename)
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Tabela salva: {path}")
    return path


def short_quadrant_label(label):
    return QUADRANT_SHORT_LABEL.get(str(label), str(label))


def wrap_text(text, width=38):
    text = str(text)
    if len(text) <= width:
        return text
    words = text.replace("__", "__ ").replace("_", "_ ").split()
    lines, current = [], ""
    for word in words:
        if len(current) + len(word) + 1 > width:
            lines.append(current.strip())
            current = word
        else:
            current += " " + word
    if current:
        lines.append(current.strip())
    return "<br>".join(lines)


def apply_tcc_layout(fig, title=None, x_title=None, y_title=None, legend_title=None, height=None, width=None):
    fig.update_layout(
        template=PLOT_TEMPLATE,
        title={"text": title or fig.layout.title.text, "x": 0.02, "xanchor": "left"},
        font={"family": "Arial", "size": 14},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
        margin={"l": 80, "r": 40, "t": 95, "b": 85},
        width=width or FIG_WIDTH,
        height=height or FIG_HEIGHT,
    )
    if x_title is not None:
        fig.update_xaxes(title_text=x_title, showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    if y_title is not None:
        fig.update_yaxes(title_text=y_title, showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    if legend_title is not None:
        fig.update_layout(legend_title_text=legend_title)
    return fig


def save_fig(fig, filename, title=None, x_title=None, y_title=None, legend_title=None, height=None, width=None):
    """Salva HTML, PNG e SVG com layout padronizado para TCC.

    O HTML preserva interatividade; PNG/SVG servem para monografia e relatório.
    """
    fig = apply_tcc_layout(fig, title=title, x_title=x_title, y_title=y_title, legend_title=legend_title, height=height, width=width)
    html_path = os.path.join(FIGURES_PATH, f"{filename}.html")
    fig.write_html(
        html_path,
        include_plotlyjs="cdn",
        config={"displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": FIG_SCALE}},
    )
    print(f"Figura HTML salva: {html_path}")

    if EXPORT_PNG:
        try:
            png_path = os.path.join(FIGURES_PATH, f"{filename}.png")
            fig.write_image(png_path, scale=FIG_SCALE, width=width or FIG_WIDTH, height=height or FIG_HEIGHT)
            print(f"Figura PNG salva: {png_path}")
        except Exception as e:
            print(f"[AVISO] PNG não salvo para {filename}. Instale/atualize kaleido. Erro: {e}")

    if EXPORT_SVG:
        try:
            svg_path = os.path.join(FIGURES_PATH, f"{filename}.svg")
            fig.write_image(svg_path, width=width or FIG_WIDTH, height=height or FIG_HEIGHT)
            print(f"Figura SVG salva: {svg_path}")
        except Exception as e:
            print(f"[AVISO] SVG não salvo para {filename}. Instale/atualize kaleido. Erro: {e}")

    if SHOW_FIGURES:
        fig.show()
    return html_path


def save_plot_index():
    rows = []
    for ext in ["html", "png", "svg"]:
        for path in sorted(Path(FIGURES_PATH).glob(f"*.{ext}")):
            rows.append({"arquivo": path.name, "tipo": ext, "caminho": str(path)})
    if rows:
        return save_table(pd.DataFrame(rows), "figure_inventory.csv")
    return None

def read_csv_smart(path, **kwargs):
    attempts = [
        {"sep": ";"},
        {"sep": ","},
        {"sep": "\t"},
        {"sep": None, "engine": "python"},
    ]
    last_error = None
    for params in attempts:
        try:
            df_tmp = pd.read_csv(path, **{**params, **kwargs})
            if df_tmp.shape[1] > 1:
                df_tmp.columns = [str(c).strip() for c in df_tmp.columns]
                return df_tmp
        except Exception as e:
            last_error = e
    raise ValueError(f"Não foi possível ler {path}. Último erro: {last_error}")


def read_table_smart(path):
    path = str(path)
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path)
    if path.lower().endswith((".csv", ".txt")):
        return read_csv_smart(path)
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    raise ValueError(f"Formato não suportado: {path}")


def extract_song_id_from_filename(filename):
    base = os.path.basename(filename)
    match = re.search(r"(\d+)", base)
    return int(match.group(1)) if match else np.nan


def find_time_column(df_in):
    candidates = ["frameTime", "frame_time", "time", "Time", "timestamp", "seconds", "time_sec"]
    for c in candidates:
        if c in df_in.columns:
            return c
    numeric_cols = df_in.select_dtypes(include=[np.number]).columns.tolist()
    for c in numeric_cols:
        if "time" in str(c).lower():
            return c
    raise ValueError("Não foi possível identificar a coluna de tempo.")


def sanitize_column_name(col):
    return (
        str(col)
        .replace("[", "_")
        .replace("]", "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace(",", "_")
        .replace(" ", "_")
    )


def sanitize_column_names(df_in):
    df_out = df_in.copy()
    df_out.columns = [sanitize_column_name(c) for c in df_out.columns]
    return df_out


## 3. Carregamento e alinhamento dos dados

Se `ALIGNED_DATA_PATH` estiver preenchido, o notebook usa esse arquivo. Caso contrário, carrega as features openSMILE e alinha com os rótulos dinâmicos de `valence` e `arousal`.


In [5]:
# =========================
# CARREGAMENTO — opensmile_blocks.parquet
# O parquet já contém uma linha por bloco de 10s com:
#   - features : os_mean__* (média dos 20 frames) e os_std__* (desvio padrão)
#   - labels   : valence, arousal, quadrant_label (threshold=0, escala -1..1)
#   - meta     : song_id, filename, block_id, block_start_sec, block_end_sec,
#                block_duration_sec, quadrant, method
# Gerado pelo notebook 00_construir_opensmile_blocks.ipynb
# =========================
from pathlib import Path as _Path

_blocks_path = _Path(OPENSMILE_BLOCKS_PATH)
if not _blocks_path.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {_blocks_path}\n"
        "Execute o notebook 00_construir_opensmile_blocks.ipynb primeiro."
    )

df = pd.read_parquet(_blocks_path)
df[SONG_ID_COL] = df[SONG_ID_COL].astype(str)

_n_mean = sum(1 for c in df.columns if c.startswith(MEAN_PREFIX))
_n_std  = sum(1 for c in df.columns if c.startswith(STD_PREFIX))

print(f"Dataset carregado : {df.shape}")
print(f"  Músicas         : {df[SONG_ID_COL].nunique()}")
print(f"  Blocos          : {len(df)}")
print(f"  Features mean   : {_n_mean}  ({MEAN_PREFIX}*)")
print(f"  Features std    : {_n_std}   ({STD_PREFIX}*)")
print(f"  Quadrantes      :")
print(df[QUADRANT_COL].value_counts().reindex(QUADRANT_ORDER).to_string())

display(df.head())
save_table(df.head(1000), "aligned_sample_preview.csv")


Dataset carregado : (6506, 531)
  Músicas         : 1802
  Blocos          : 6506
  Features mean   : 260  (os_mean__*)
  Features std    : 260   (os_std__*)
  Quadrantes      :
quadrant_label
Q1_High_Valence_High_Arousal    3217
Q2_Low_Valence_High_Arousal     1019
Q3_Low_Valence_Low_Arousal      1382
Q4_High_Valence_Low_Arousal      888


,song_id,filename,block_id,block_start_sec,block_end_sec,block_duration_sec,valence,arousal,quadrant,quadrant_label,...,os_std__pcm_fftMag_mfcc_sma_de[10]_stddev,os_std__pcm_fftMag_mfcc_sma_de[10]_amean,os_std__pcm_fftMag_mfcc_sma_de[11]_stddev,os_std__pcm_fftMag_mfcc_sma_de[11]_amean,os_std__pcm_fftMag_mfcc_sma_de[12]_stddev,os_std__pcm_fftMag_mfcc_sma_de[12]_amean,os_std__pcm_fftMag_mfcc_sma_de[13]_stddev,os_std__pcm_fftMag_mfcc_sma_de[13]_amean,os_std__pcm_fftMag_mfcc_sma_de[14]_stddev,os_std__pcm_fftMag_mfcc_sma_de[14]_amean
0,2,2.csv,0,15.0,25.0,10.0,-0.085140,-0.137974,Q3,Q3_Low_Valence_Low_Arousal,...,0.357739,0.179645,0.442328,0.127740,0.397949,0.138626,0.296541,0.099770,0.300759,0.090531
1,2,2.csv,1,25.0,35.0,10.0,-0.241535,-0.191833,Q3,Q3_Low_Valence_Low_Arousal,...,0.364996,0.142497,0.279245,0.134915,0.261003,0.131748,0.238374,0.137918,0.279821,0.124257
2,2,2.csv,2,35.0,45.0,10.0,-0.319857,-0.262745,Q3,Q3_Low_Valence_Low_Arousal,...,0.349096,0.116154,0.306713,0.114211,0.321004,0.074885,0.313462,0.120760,0.190085,0.093429
3,3,3.csv,0,15.0,25.0,10.0,-0.208460,-0.153975,Q3,Q3_Low_Valence_Low_Arousal,...,0.361893,0.109599,0.218232,0.156757,0.162581,0.083067,0.219203,0.114533,0.192439,0.094067
4,3,3.csv,1,25.0,35.0,10.0,-0.287444,-0.178349,Q3,Q3_Low_Valence_Low_Arousal,...,0.369269,0.257942,0.203643,0.137051,0.418139,0.236478,0.337667,0.117218,0.137536,0.147779


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\aligned_sample_preview.csv


'C:\\dev\\python\\TCC Ajustado\\Dados\\comparative_outputs\\opensmile_pycaret_outputs\\tables\\aligned_sample_preview.csv'

## 4. Configura??es adicionais e cria??o dos quadrantes

Esta se??o cria a vari?vel alvo `quadrant_label`, infere os limiares adequados para a escala VA e monta o dataset de modelagem. Quando `USE_BLOCK_LEVEL=True`, as features de frame s?o agregadas por m?sica/bloco de 10s, preservando `song_id` apenas como grupo de valida??o.

In [6]:
# =========================
# CONFIGURAÇÕES ADICIONAIS
# =========================
MIN_CLASS_COUNT      = globals().get("MIN_CLASS_COUNT", 5)
SAVE_MODEL_ARTIFACTS = globals().get("SAVE_MODEL_ARTIFACTS", True)
PYCARET_SESSION_ID   = globals().get("PYCARET_SESSION_ID", RANDOM_STATE)

# Escala VA e thresholds (já aplicados no notebook 00 ao gerar o parquet)
VA_SCALE_ASSUMPTION      = "-1_to_1"
VALENCE_THRESHOLD_POLICY = 0.0
AROUSAL_THRESHOLD_POLICY = 0.0
QUADRANT_THRESHOLD_MODE  = "fixed_zero"

# Colunas de meta/label — não devem entrar como features
LEAKAGE_COLS = {
    SONG_ID_COL, "filename", BLOCK_ID_COL,
    "block_start_sec", "block_end_sec", "block_duration_sec",
    VALENCE_COL, AROUSAL_COL,
    "quadrant",    # código curto Q1/Q2/Q3/Q4 — derivado do target
    QUADRANT_COL,  # quadrant_label — o alvo
    "method",
    TIME_COL,      # frametime — não existe no parquet; auditoria de vazamento
    "time_sec", "n_frames",
}


def get_base_feature_columns(df_in):
    """Retorna apenas colunas os_mean__* e os_std__* — nenhuma coluna de meta/label."""
    return [
        c for c in df_in.columns
        if c.startswith(MEAN_PREFIX) or c.startswith(STD_PREFIX)
    ]


def build_modeling_dataset(df_in):
    """
    O parquet já está no nível de bloco com quadrant_label computado.
    Apenas garante tipos corretos e converte quadrant_label para Categorical.
    """
    data = df_in.copy()
    data[SONG_ID_COL] = data[SONG_ID_COL].astype(str)
    data[VALENCE_COL] = pd.to_numeric(data[VALENCE_COL], errors="coerce")
    data[AROUSAL_COL] = pd.to_numeric(data[AROUSAL_COL], errors="coerce")
    data = data.dropna(subset=[VALENCE_COL, AROUSAL_COL, QUADRANT_COL]).copy()
    data[QUADRANT_COL] = pd.Categorical(
        data[QUADRANT_COL], categories=QUADRANT_ORDER, ordered=True
    )
    return data


model_df = build_modeling_dataset(df)
feature_cols = get_base_feature_columns(model_df)

# Thresholds fixos (usados em gráficos de EDA)
VALENCE_THRESHOLD = VALENCE_THRESHOLD_POLICY
AROUSAL_THRESHOLD = AROUSAL_THRESHOLD_POLICY

class_counts = (
    model_df[QUADRANT_COL]
    .value_counts()
    .reindex(QUADRANT_ORDER, fill_value=0)
)
small_classes = class_counts[class_counts < MIN_CLASS_COUNT]
if not small_classes.empty:
    raise RuntimeError(f"Classes com menos de {MIN_CLASS_COUNT} amostras: {small_classes.to_dict()}")

class_balance = class_counts.rename_axis(QUADRANT_COL).reset_index(name="n")
class_balance["percent"] = class_balance["n"] / class_balance["n"].sum() * 100
save_table(class_balance, "class_balance_quadrants.csv")

_n_mean_feat = sum(1 for c in feature_cols if c.startswith(MEAN_PREFIX))
_n_std_feat  = sum(1 for c in feature_cols if c.startswith(STD_PREFIX))
print(f"Dataset de modelagem : {model_df.shape}")
print(f"  Features mean ({MEAN_PREFIX}): {_n_mean_feat}")
print(f"  Features std  ({STD_PREFIX}) : {_n_std_feat}")
print(f"  Total features              : {len(feature_cols)}")
print(f"  Política VA : {VA_SCALE_ASSUMPTION} | threshold={VALENCE_THRESHOLD:.1f}")
display(class_counts.rename("n").to_frame())
display(class_balance)
save_table(model_df, "quadrant_modeling_dataset.csv")


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\class_balance_quadrants.csv
Dataset de modelagem : (6506, 531)
  Features mean (os_mean__): 260
  Features std  (os_std__) : 260
  Total features              : 520
  Política VA : -1_to_1 | threshold=0.0


,n
quadrant_label,
Q1_High_Valence_High_Arousal,3217
Q2_Low_Valence_High_Arousal,1019
Q3_Low_Valence_Low_Arousal,1382
Q4_High_Valence_Low_Arousal,888


,quadrant_label,n,percent
0,Q1_High_Valence_High_Arousal,3217,49.446665
1,Q2_Low_Valence_High_Arousal,1019,15.662465
2,Q3_Low_Valence_Low_Arousal,1382,21.241931
3,Q4_High_Valence_Low_Arousal,888,13.648939


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\quadrant_modeling_dataset.csv


'C:\\dev\\python\\TCC Ajustado\\Dados\\comparative_outputs\\opensmile_pycaret_outputs\\tables\\quadrant_modeling_dataset.csv'

## 5. EDA, invent?rio e ranking de features

Esta se??o exporta balan?o de classes, gr?ficos de `valence`/`arousal`, invent?rio de features, diagn?stico de qualidade e ranking estat?stico das features candidatas.

In [7]:
def safe_sample(df_in, max_rows=MAX_SAMPLE_FOR_EDA, random_state=RANDOM_STATE):
    if len(df_in) <= max_rows:
        return df_in.copy()
    return df_in.sample(max_rows, random_state=random_state).copy()


def feature_category(feature_name):
    name = feature_name.lower()
    if any(token in name for token in ["f0", "pitch", "voicing"]):
        return "Melody / Pitch"
    if any(token in name for token in ["energy", "rms", "loudness", "intensity"]):
        return "Dynamics"
    if any(token in name for token in ["zcr", "tempo", "rhythm", "onset", "duration"]):
        return "Rhythm / Temporal"
    if any(token in name for token in ["jitter", "shimmer", "hnr"]):
        return "Expressivity"
    if any(token in name for token in ["mfcc", "audspec", "spectral", "fft", "harmonicity", "sharpness", "rasta"]):
        return "Timbre / Tone Colour"
    return "Unmapped / Other"


def build_feature_inventory(feature_columns):
    """
    Constrói inventário com base_feature e aggregation corretos para os prefixos
    os_mean__* e os_std__* gerados pelo notebook 00_construir_opensmile_blocks.
    """
    rows = []
    for feature in feature_columns:
        if feature.startswith(MEAN_PREFIX):
            aggregation = "mean"
            base = feature[len(MEAN_PREFIX):]
        elif feature.startswith(STD_PREFIX):
            aggregation = "std"
            base = feature[len(STD_PREFIX):]
        else:
            parts = feature.split("__")
            aggregation = parts[-1] if len(parts) > 1 else "raw"
            base = parts[0]
        rows.append({
            "feature": feature,
            "base_feature": base,
            "aggregation": aggregation,
            "categoria_musical": feature_category(base),
        })
    return pd.DataFrame(rows)


def build_feature_quality(df_in, feature_columns):
    rows = []
    for feature in feature_columns:
        s = pd.to_numeric(df_in[feature], errors="coerce")
        rows.append({
            "feature": feature,
            "missing_rate": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna=True)),
            "variance": float(s.var(skipna=True)) if s.notna().sum() > 1 else np.nan,
            "mean": float(s.mean(skipna=True)) if s.notna().any() else np.nan,
            "std": float(s.std(skipna=True)) if s.notna().sum() > 1 else np.nan,
            "min": float(s.min(skipna=True)) if s.notna().any() else np.nan,
            "max": float(s.max(skipna=True)) if s.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def select_usable_features(quality_df):
    usable = quality_df[
        quality_df["missing_rate"].le(MAX_MISSING_RATE)
        & quality_df["n_unique"].gt(1)
        & quality_df["variance"].fillna(0).gt(0)
    ]["feature"].tolist()
    if not usable:
        raise RuntimeError("Nenhuma feature passou pelos filtros de qualidade.")
    return usable


# Balanço de classes
class_balance = (
    model_df[QUADRANT_COL]
    .value_counts()
    .reindex(QUADRANT_ORDER, fill_value=0)
    .rename_axis(QUADRANT_COL)
    .reset_index(name="n")
)
class_balance["percent"] = class_balance["n"] / class_balance["n"].sum() * 100
save_table(class_balance, "class_balance_quadrants.csv")

fig_balance = px.bar(
    class_balance,
    x=QUADRANT_COL,
    y="n",
    text=class_balance["percent"].map(lambda x: f"{x:.1f}%"),
    title="Distribuição dos quadrantes emocionais",
)
fig_balance.update_layout(xaxis_title="Quadrante", yaxis_title="Amostras")
fig_balance.update_traces(textposition="outside")
save_fig(fig_balance, "class_balance_quadrants")

# Espaço valence-arousal
eda_sample = safe_sample(model_df)
fig_scatter = px.scatter(
    eda_sample,
    x=VALENCE_COL,
    y=AROUSAL_COL,
    color=QUADRANT_COL,
    color_discrete_map=QUADRANT_COLOR_MAP,
    category_orders={QUADRANT_COL: QUADRANT_ORDER},
    opacity=0.55,
    title="Espaço Valence-Arousal por quadrante",
)
fig_scatter.add_vline(x=VALENCE_THRESHOLD, line_dash="dash", line_color="black")
fig_scatter.add_hline(y=AROUSAL_THRESHOLD, line_dash="dash", line_color="black")
save_fig(fig_scatter, "scatter_valence_arousal_quadrants")

fig_valence = px.histogram(
    eda_sample, x=VALENCE_COL, color=QUADRANT_COL,
    color_discrete_map=QUADRANT_COLOR_MAP,
    category_orders={QUADRANT_COL: QUADRANT_ORDER},
    nbins=60, barmode="overlay",
    title="Distribuição de valence por quadrante",
)
fig_valence.update_traces(opacity=0.65)
save_fig(fig_valence, "hist_valence_by_quadrant")

fig_arousal = px.histogram(
    eda_sample, x=AROUSAL_COL, color=QUADRANT_COL,
    color_discrete_map=QUADRANT_COLOR_MAP,
    category_orders={QUADRANT_COL: QUADRANT_ORDER},
    nbins=60, barmode="overlay",
    title="Distribuição de arousal por quadrante",
)
fig_arousal.update_traces(opacity=0.65)
save_fig(fig_arousal, "hist_arousal_by_quadrant")

# Inventário e qualidade de features
feature_inventory = build_feature_inventory(feature_cols)
feature_quality   = build_feature_quality(model_df, feature_cols)
usable_feature_cols = select_usable_features(feature_quality)
feature_inventory["usable"] = feature_inventory["feature"].isin(usable_feature_cols)

save_table(feature_inventory, "classification_feature_inventory.csv")
save_table(feature_quality,   "feature_quality_quadrants.csv")

category_summary = (
    feature_inventory
    .groupby("categoria_musical", as_index=False)
    .agg(n_features=("feature", "count"), n_usable=("usable", "sum"))
    .sort_values("n_usable", ascending=False)
)
save_table(category_summary, "feature_category_summary_quadrants.csv")

fig_category = px.bar(
    category_summary, x="categoria_musical", y="n_usable",
    title="Features utilizáveis por categoria musical",
)
fig_category.update_layout(xaxis_title="Categoria", yaxis_title="Features utilizáveis")
save_fig(fig_category, "feature_category_summary_quadrants")

# Ranking estatístico das features
X_rank = model_df[usable_feature_cols].replace([np.inf, -np.inf], np.nan)
y_rank = model_df[QUADRANT_COL].astype(str)
X_rank_imputed = SimpleImputer(strategy="median").fit_transform(X_rank)

anova_scores, anova_pvalues = f_classif(X_rank_imputed, y_rank)
try:
    mi_scores = mutual_info_classif(X_rank_imputed, y_rank, random_state=RANDOM_STATE)
except Exception as e:
    print(f"[AVISO] mutual_info_classif não calculado: {e}")
    mi_scores = np.full(len(usable_feature_cols), np.nan)

feature_ranking = pd.DataFrame({
    "feature": usable_feature_cols,
    "anova_f": anova_scores,
    "anova_pvalue": anova_pvalues,
    "mutual_info": mi_scores,
}).merge(feature_inventory, on="feature", how="left")
feature_ranking = feature_ranking.sort_values("anova_f", ascending=False)
save_table(feature_ranking, "feature_ranking_quadrants.csv")

fig_top_features = px.bar(
    feature_ranking.head(25).sort_values("anova_f"),
    x="anova_f", y="feature", orientation="h",
    color="categoria_musical",
    title="Top 25 features por ANOVA F-score",
)
fig_top_features.update_layout(xaxis_title="ANOVA F-score", yaxis_title="Feature")
save_fig(fig_top_features, "top25_features_quadrants")

print(f"Features candidatas             : {len(feature_cols)}")
print(f"  - mean ({MEAN_PREFIX})     : {sum(1 for c in feature_cols if c.startswith(MEAN_PREFIX))}")
print(f"  - std  ({STD_PREFIX})      : {sum(1 for c in feature_cols if c.startswith(STD_PREFIX))}")
print(f"Features utilizáveis (qualidade): {len(usable_feature_cols)}")
display(category_summary)
display(feature_ranking.head(15))


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\class_balance_quadrants.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\class_balance_quadrants.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\class_balance_quadrants.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\class_balance_quadrants.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\scatter_valence_arousal_quadrants.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\scatter_valence_arousal_quadrants.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\scatter_valence_arousal_quadrants.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_valence_by_quadrant.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_valence_by_quadrant.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_valence_by_quadrant.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_arousal_by_quadrant.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_arousal_by_quadrant.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\hist_arousal_by_quadrant.svg


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_feature_inventory.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\feature_quality_quadrants.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\feature_category_summary_quadrants.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\feature_category_summary_quadrants.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\feature_category_summary_quadrants.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\feature_category_summary_quadrants.svg


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\feature_ranking_quadrants.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\top25_features_quadrants.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\top25_features_quadrants.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\top25_features_quadrants.svg


Features candidatas             : 520
  - mean (os_mean__)     : 260
  - std  (os_std__)      : 260
Features utilizáveis (qualidade): 520


,categoria_musical,n_features,n_usable
4,Timbre / Tone Colour,456,456
1,Expressivity,32,32
2,Melody / Pitch,16,16
0,Dynamics,8,8
3,Rhythm / Temporal,8,8


,feature,anova_f,anova_pvalue,mutual_info,base_feature,aggregation,categoria_musical,usable
103,os_mean__pcm_fftMag_mfcc_sma[1]_amean,607.439092,0.000000e+00,0.142299,pcm_fftMag_mfcc_sma[1]_amean,mean,Timbre / Tone Colour,True
13,os_mean__audspec_lengthL1norm_sma_amean,455.081072,2.113547e-268,0.137660,audspec_lengthL1norm_sma_amean,mean,Timbre / Tone Colour,True
10,os_mean__logHNR_sma_stddev,390.606018,2.780432e-233,0.104043,logHNR_sma_stddev,mean,Expressivity,True
89,os_mean__pcm_fftMag_spectralEntropy_sma_amean,351.169500,2.298858e-211,0.079647,pcm_fftMag_spectralEntropy_sma_amean,mean,Timbre / Tone Colour,True
142,os_mean__audspec_lengthL1norm_sma_de_stddev,302.524822,7.426909e-184,0.100688,audspec_lengthL1norm_sma_de_stddev,mean,Timbre / Tone Colour,True
140,os_mean__logHNR_sma_de_stddev,300.622615,9.066131e-183,0.100954,logHNR_sma_de_stddev,mean,Expressivity,True
19,os_mean__pcm_zcr_sma_amean,297.883062,3.340561e-181,0.112146,pcm_zcr_sma_amean,mean,Rhythm / Temporal,True
83,os_mean__pcm_fftMag_spectralRollOff90.0_sma_amean,282.820348,1.469489e-172,0.106279,pcm_fftMag_spectralRollOff90.0_sma_amean,mean,Timbre / Tone Colour,True
85,os_mean__pcm_fftMag_spectralFlux_sma_amean,266.829172,2.517876e-163,0.116508,pcm_fftMag_spectralFlux_sma_amean,mean,Timbre / Tone Colour,True
212,os_mean__pcm_fftMag_spectralRollOff90.0_sma_de...,232.303704,3.494390e-143,0.107095,pcm_fftMag_spectralRollOff90.0_sma_de_stddev,mean,Timbre / Tone Colour,True


## 5.1a Contribuição de cada família acústica openSMILE

Esta seção calcula o poder discriminativo médio (ANOVA F-score) de cada categoria de feature.
O objetivo é identificar **quais famílias acústicas** (timbre, dinâmica, ritmo, pitch, expressividade)
mais contribuem para separar os quadrantes emocionais no espaço Valence–Arousal.

In [8]:
# =========================
# ANÁLISE POR CATEGORIA — ANOVA F-score médio por família acústica
# =========================

if "feature_ranking" in dir() and not feature_ranking.empty and "categoria_musical" in feature_ranking.columns:
    cat_anova = (
        feature_ranking.groupby("categoria_musical")
        .agg(
            n_features=("feature", "count"),
            anova_f_mean=("anova_f", "mean"),
            anova_f_max=("anova_f", "max"),
        )
        .reset_index()
        .sort_values("anova_f_mean", ascending=False)
    )
    usable_cat = (
        feature_ranking[feature_ranking["feature"].isin(usable_feature_cols)]
        .groupby("categoria_musical")
        .agg(n_usable=("feature", "count"))
        .reset_index()
    )
    cat_anova = cat_anova.merge(usable_cat, on="categoria_musical", how="left")
    cat_anova["n_usable"] = cat_anova["n_usable"].fillna(0).astype(int)

    save_table(cat_anova, "feature_category_anova_summary.csv")

    display(Markdown("### Ranking de famílias acústicas por poder discriminativo (ANOVA F-score médio)"))
    display(cat_anova.rename(columns={
        "categoria_musical": "Categoria",
        "n_features": "Nº features",
        "n_usable": "Nº usáveis",
        "anova_f_mean": "ANOVA F médio",
        "anova_f_max": "ANOVA F máximo",
    }))

    fig = px.bar(
        cat_anova.sort_values("anova_f_mean"),
        x="anova_f_mean",
        y="categoria_musical",
        orientation="h",
        text=cat_anova.sort_values("anova_f_mean")["anova_f_mean"].map(lambda x: f"{x:.1f}"),
        color="categoria_musical",
        title="Poder discriminativo por categoria acústica (ANOVA F-score médio)",
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(showlegend=False)
    save_fig(fig, "tcc_05a_feature_category_anova",
             x_title="ANOVA F-score médio", y_title="Categoria acústica")

    top_cat = cat_anova.iloc[0]
    display(Markdown(
        f"\n**Interpretação:** A categoria mais discriminativa é "
        f"**{top_cat['categoria_musical']}** "
        f"(ANOVA F médio = {top_cat['anova_f_mean']:.2f}), com {top_cat['n_usable']} features usáveis. "
        f"Isso indica que essa família apresenta maior separabilidade entre os quadrantes emocionais."
    ))
else:
    print("[AVISO] feature_ranking não disponível. Execute a seção 5 (EDA) antes desta célula.")


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\feature_category_anova_summary.csv


### Ranking de famílias acústicas por poder discriminativo (ANOVA F-score médio)

,Categoria,Nº features,ANOVA F médio,ANOVA F máximo,Nº usáveis
0,Expressivity,32,103.432434,390.606018,32
1,Rhythm / Temporal,8,100.609302,297.883062,8
2,Dynamics,8,56.420002,150.269859,8
3,Melody / Pitch,16,54.845311,163.962928,16
4,Timbre / Tone Colour,456,41.173874,607.439092,456


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05a_feature_category_anova.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05a_feature_category_anova.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05a_feature_category_anova.svg



**Interpretação:** A categoria mais discriminativa é **Expressivity** (ANOVA F médio = 103.43), com 32 features usáveis. Isso indica que essa família apresenta maior separabilidade entre os quadrantes emocionais.

## 5.1 Gráficos finais para o TCC

Esta seção gera versões mais legíveis dos gráficos principais. A ideia é usar estes arquivos no relatório/TCC: distribuição dos quadrantes, histogramas com limiares, espaço Valence–Arousal com quadrantes identificados e ranking de features com nomes encurtados.


In [9]:
# Versões finais dos gráficos de EDA para relatório/TCC
plot_df = model_df.copy()
plot_df["quadrant_short"] = plot_df[QUADRANT_COL].astype(str).map(short_quadrant_label)
plot_df_sample = safe_sample(plot_df, max_rows=MAX_SAMPLE_FOR_EDA, random_state=RANDOM_STATE)

# 1) Distribuição de classes com percentual e linha de equilíbrio ideal
class_balance_plot = class_balance.copy()
class_balance_plot["quadrant_short"] = class_balance_plot[QUADRANT_COL].map(short_quadrant_label)
class_balance_plot["percent_label"] = class_balance_plot["percent"].map(lambda x: f"{x:.1f}%")
fig = px.bar(
    class_balance_plot,
    x="quadrant_short",
    y="percent",
    text="percent_label",
    color=QUADRANT_COL,
    color_discrete_map=QUADRANT_COLOR_MAP,
    title="Distribuição dos quadrantes emocionais",
)
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="Distribuição ideal: 25%")
fig.update_traces(textposition="outside", hovertemplate="Quadrante=%{x}<br>Percentual=%{y:.2f}%<extra></extra>")
save_fig(fig, "tcc_01_class_balance_percent", x_title="Quadrante emocional", y_title="Percentual dos blocos (%)", legend_title="Quadrante")

# 2) Espaço VA com divisões e rótulos dos quadrantes
fig = px.scatter(
    plot_df_sample,
    x=VALENCE_COL,
    y=AROUSAL_COL,
    color=QUADRANT_COL,
    color_discrete_map=QUADRANT_COLOR_MAP,
    category_orders={QUADRANT_COL: QUADRANT_ORDER},
    opacity=0.48,
    title="Distribuição dos blocos no espaço Valence–Arousal",
    hover_data=[SONG_ID_COL, QUADRANT_COL],
)
fig.add_vline(x=VALENCE_THRESHOLD, line_dash="dash", line_color="black", annotation_text="limiar valence")
fig.add_hline(y=AROUSAL_THRESHOLD, line_dash="dash", line_color="black", annotation_text="limiar arousal")
# posicionamento aproximado dos rótulos nas quatro regiões
x_min, x_max = plot_df_sample[VALENCE_COL].min(), plot_df_sample[VALENCE_COL].max()
y_min, y_max = plot_df_sample[AROUSAL_COL].min(), plot_df_sample[AROUSAL_COL].max()
x_left = x_min + 0.12 * (x_max - x_min); x_right = x_max - 0.12 * (x_max - x_min)
y_low = y_min + 0.12 * (y_max - y_min); y_high = y_max - 0.12 * (y_max - y_min)
for label, x_pos, y_pos in [
    ("Q1 Alegre/Energético", x_right, y_high),
    ("Q2 Tenso/Raivoso", x_left, y_high),
    ("Q3 Triste/Melancólico", x_left, y_low),
    ("Q4 Calmo/Relaxado", x_right, y_low),
]:
    fig.add_annotation(x=x_pos, y=y_pos, text=label, showarrow=False, bgcolor="rgba(255,255,255,0.75)", bordercolor="rgba(0,0,0,0.2)")
save_fig(fig, "tcc_02_va_space_quadrants", x_title="Valence", y_title="Arousal", legend_title="Quadrante")

# 3) Histogramas com limiares explícitos
for target_col, threshold, pretty_name in [(VALENCE_COL, VALENCE_THRESHOLD, "Valence"), (AROUSAL_COL, AROUSAL_THRESHOLD, "Arousal")]:
    fig = px.histogram(
        plot_df_sample,
        x=target_col,
        color=QUADRANT_COL,
        color_discrete_map=QUADRANT_COLOR_MAP,
        category_orders={QUADRANT_COL: QUADRANT_ORDER},
        nbins=50,
        barmode="overlay",
        marginal="box",
        histnorm="percent",
        title=f"Distribuição de {pretty_name} por quadrante",
    )
    fig.update_traces(opacity=0.62)
    fig.add_vline(x=threshold, line_dash="dash", line_color="black", annotation_text=f"limiar = {threshold:.2f}")
    save_fig(fig, f"tcc_03_hist_{target_col}_threshold", x_title=pretty_name, y_title="Percentual (%)", legend_title="Quadrante")

# 4) Ranking de features com nomes legíveis
rank_plot = feature_ranking.head(25).copy().sort_values("anova_f")
rank_plot["feature_label"] = rank_plot["feature"].map(lambda x: wrap_text(x, width=42))
fig = px.bar(
    rank_plot,
    x="anova_f",
    y="feature_label",
    orientation="h",
    color="categoria_musical",
    title="Top 25 features associadas aos quadrantes (ANOVA F-score)",
    hover_data={"feature": True, "anova_f": ":.3f", "mutual_info": ":.4f", "feature_label": False},
)
save_fig(fig, "tcc_04_top25_features_anova", x_title="ANOVA F-score", y_title="Feature", legend_title="Categoria musical", height=900)

# 5) Distribuição por categoria musical das features úteis
cat_plot = category_summary.sort_values("n_usable", ascending=True).copy()
fig = px.bar(
    cat_plot,
    x="n_usable",
    y="categoria_musical",
    orientation="h",
    text="n_usable",
    title="Quantidade de features úteis por categoria musical",
    hover_data=["n_features", "n_usable"],
)
fig.update_traces(textposition="outside")
save_fig(fig, "tcc_05_feature_category_summary", x_title="Número de features úteis", y_title="Categoria musical", height=650)


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_01_class_balance_percent.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_01_class_balance_percent.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_01_class_balance_percent.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_02_va_space_quadrants.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_02_va_space_quadrants.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_02_va_space_quadrants.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_valence_threshold.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_valence_threshold.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_valence_threshold.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_arousal_threshold.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_arousal_threshold.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_03_hist_arousal_threshold.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_04_top25_features_anova.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_04_top25_features_anova.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_04_top25_features_anova.svg


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05_feature_category_summary.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05_feature_category_summary.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_05_feature_category_summary.svg


'C:\\dev\\python\\TCC Ajustado\\Dados\\comparative_outputs\\opensmile_pycaret_outputs\\figures\\tcc_05_feature_category_summary.html'

## 6. Valida??o oficial com GroupKFold por `song_id`

A avalia??o abaixo ? a refer?ncia metodol?gica do notebook. Ela evita vazamento entre treino e teste usando grupos por m?sica e seleciona o melhor modelo por `macro_f1_mean`.

In [10]:
def build_candidate_models():
    # Conjunto enxuto para quadrantes: prioriza modelos compatíveis com class_weight
    # e evita famílias muito lentas como GradientBoosting/AdaBoost para o benchmark principal.
    return {
        "Dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "Dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        "LogisticRegression_balanced": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "RidgeClassifier_balanced": RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=250,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "ExtraTrees_balanced": ExtraTreesClassifier(
            n_estimators=250,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "KNeighbors": KNeighborsClassifier(n_neighbors=15, weights="distance"),
        "DecisionTree_balanced": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "GaussianNB": GaussianNB(),
    }


def build_pipeline(estimator, selector_k=None, n_features=None):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if selector_k is not None:
        k_effective = min(int(selector_k), int(n_features))
        steps.append(("selector", SelectKBest(score_func=f_classif, k=k_effective)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def normalized_confusion_matrix_df(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    return pd.DataFrame(cm, index=labels, columns=labels)


def evaluate_pipeline_groupkfold(pipeline, X, y, groups, labels, cv):
    fold_rows = []
    oof_pred = pd.Series(index=y.index, dtype="object")

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        model = clone(pipeline)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        oof_pred.iloc[test_idx] = pred

        row = {
            "fold": fold_idx,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_groups_train": pd.Series(groups.iloc[train_idx]).nunique(),
            "n_groups_test": pd.Series(groups.iloc[test_idx]).nunique(),
        }
        row.update(classification_metrics(y_test, pred))
        fold_rows.append(row)

    return pd.DataFrame(fold_rows), oof_pred


def summarize_fold_metrics(fold_df):
    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]
    summary = {}
    for metric in metric_cols:
        summary[f"{metric}_mean"] = fold_df[metric].mean()
        summary[f"{metric}_std"] = fold_df[metric].std(ddof=1)
    return summary


X = model_df[usable_feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df[QUADRANT_COL].astype(str)
groups = model_df[SONG_ID_COL].astype(str)
labels_present = [label for label in QUADRANT_ORDER if label in set(y)]

n_groups = groups.nunique()
cv_splits = min(N_SPLITS, n_groups)
if cv_splits < 2:
    raise RuntimeError("S?o necess?rios pelo menos 2 grupos de song_id para GroupKFold.")
cv = GroupKFold(n_splits=cv_splits)

# Benchmark principal: conjuntos pequenos para não travar a validação oficial.
selector_options = [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]
models = build_candidate_models()

results_rows = []
fold_metric_frames = []
errors = []

for model_name, estimator in tqdm(models.items(), desc="Modelos sklearn"):
    for selector_k in selector_options:
        selector_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
        pipeline = build_pipeline(estimator, selector_k=selector_k, n_features=len(usable_feature_cols))
        try:
            fold_df, _ = evaluate_pipeline_groupkfold(pipeline, X, y, groups, labels_present, cv)
            fold_df.insert(0, "selector", selector_name)
            fold_df.insert(0, "model", model_name)
            fold_metric_frames.append(fold_df)

            row = {
                "model": model_name,
                "selector": selector_name,
                "n_samples": len(model_df),
                "n_groups": n_groups,
                "n_features_requested": len(usable_feature_cols),
                "n_features_effective": len(usable_feature_cols) if selector_k is None else min(selector_k, len(usable_feature_cols)),
                "cv": f"GroupKFold({cv_splits}) por song_id",
                "status": "ok",
            }
            row.update(summarize_fold_metrics(fold_df))
            results_rows.append(row)
        except Exception as e:
            error_row = {
                "model": model_name,
                "selector": selector_name,
                "n_samples": len(model_df),
                "n_groups": n_groups,
                "n_features_requested": len(usable_feature_cols),
                "n_features_effective": len(usable_feature_cols) if selector_k is None else min(selector_k, len(usable_feature_cols)),
                "cv": f"GroupKFold({cv_splits}) por song_id",
                "status": f"error: {type(e).__name__}: {e}",
            }
            results_rows.append(error_row)
            errors.append(error_row)

classification_results = pd.DataFrame(results_rows)
classification_results = classification_results.sort_values(
    ["status", "macro_f1_mean"],
    ascending=[False, False],
    na_position="last",
)
fold_metrics = pd.concat(fold_metric_frames, ignore_index=True) if fold_metric_frames else pd.DataFrame()

save_table(classification_results, "classification_groupkfold_results.csv")
if not fold_metrics.empty:
    save_table(fold_metrics, "classification_groupkfold_fold_metrics.csv")

ok_results = classification_results[classification_results["status"].eq("ok")].copy()
if ok_results.empty:
    raise RuntimeError("Nenhum modelo sklearn foi avaliado com sucesso.")

best_row = ok_results.sort_values("macro_f1_mean", ascending=False).iloc[0]
BEST_MODEL_NAME = best_row["model"]
BEST_SELECTOR_NAME = best_row["selector"]
BEST_SELECTOR_K = None if BEST_SELECTOR_NAME == "none" else int(re.search(r"k=(\d+)", BEST_SELECTOR_NAME).group(1))

best_pipeline = build_pipeline(
    build_candidate_models()[BEST_MODEL_NAME],
    selector_k=BEST_SELECTOR_K,
    n_features=len(usable_feature_cols),
)
best_fold_metrics, best_oof_pred = evaluate_pipeline_groupkfold(best_pipeline, X, y, groups, labels_present, cv)

best_predictions = model_df[[SONG_ID_COL, VALENCE_COL, AROUSAL_COL, QUADRANT_COL]].copy()
if "block_id" in model_df.columns:
    best_predictions["block_id"] = model_df["block_id"].values
if "block_start_sec" in model_df.columns:
    best_predictions["block_start_sec"] = model_df["block_start_sec"].values
best_predictions["prediction"] = best_oof_pred.values
best_predictions["correct"] = best_predictions[QUADRANT_COL].astype(str).eq(best_predictions["prediction"].astype(str))
save_table(best_predictions, "classification_oof_predictions_best_model.csv")

report = classification_report(
    y,
    best_oof_pred.astype(str),
    labels=labels_present,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
save_table(report_df, "classification_report_best_model.csv")

cm = confusion_matrix(y, best_oof_pred.astype(str), labels=labels_present)
cm_df = pd.DataFrame(cm, index=labels_present, columns=labels_present)
cm_export = cm_df.reset_index().rename(columns={"index": "actual"})
save_table(cm_export, "confusion_matrix_best_model.csv")

cm_norm_df = normalized_confusion_matrix_df(y, best_oof_pred.astype(str), labels_present)
save_table(cm_norm_df.reset_index().rename(columns={"index": "actual"}), "confusion_matrix_normalized_best_model.csv")

fig_cm = px.imshow(
    cm_df,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title=f"Matriz de confusão OOF - {BEST_MODEL_NAME} / {BEST_SELECTOR_NAME}",
)
fig_cm.update_layout(xaxis_title="Predito", yaxis_title="Real")
save_fig(fig_cm, "confusion_matrix_best_model")

fig_cm_norm = px.imshow(
    cm_norm_df,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title=f"Matriz de confusão normalizada OOF - {BEST_MODEL_NAME} / {BEST_SELECTOR_NAME}",
)
fig_cm_norm.update_layout(xaxis_title="Predito", yaxis_title="Real")
save_fig(fig_cm_norm, "confusion_matrix_normalized_best_model")

fig_results = px.bar(
    ok_results.head(25).sort_values("macro_f1_mean"),
    x="macro_f1_mean",
    y="model",
    color="selector",
    orientation="h",
    title="Top modelos por macro-F1 - GroupKFold por song_id",
)
fig_results.update_layout(xaxis_title="Macro-F1 médio", yaxis_title="Modelo")
save_fig(fig_results, "classification_macro_f1_groupkfold")

# Ajuste final do melhor pipeline em todos os dados para uso posterior.
FINAL_SKLEARN_MODEL = clone(best_pipeline).fit(X, y)
if SAVE_MODEL_ARTIFACTS:
    try:
        import joblib
        sklearn_model_path = os.path.join(MODELS_PATH, "best_sklearn_groupkfold_model.joblib")
        joblib.dump({
            "model": FINAL_SKLEARN_MODEL,
            "features": usable_feature_cols,
            "best_model_name": BEST_MODEL_NAME,
            "best_selector": BEST_SELECTOR_NAME,
            "quadrant_order": QUADRANT_ORDER,
            "valence_threshold": VALENCE_THRESHOLD,
            "arousal_threshold": AROUSAL_THRESHOLD,
            "va_scale_assumption": VA_SCALE_ASSUMPTION,
        }, sklearn_model_path)
        print("Modelo sklearn salvo:", sklearn_model_path)
    except Exception as e:
        print(f"[AVISO] N?o foi poss?vel salvar o modelo sklearn: {e}")

print("Melhor modelo oficial:", BEST_MODEL_NAME)
print("Seletor:", BEST_SELECTOR_NAME)
display(best_row.to_frame().T)
display(report_df)


Modelos sklearn:  67%|██████▋   | 6/9 [06:36<03:03, 61.09s/it]   File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Modelos sklearn: 100%|██████████| 9/9 [08:42<00:00, 58.04s/it]


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_groupkfold_results.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_groupkfold_fold_metrics.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_oof_predictions_best_model.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_report_best_model.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\confusion_matrix_best_model.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\confusion_matrix_normalized_best_model.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\confusion_matrix_best_model.html


Resorting to unclean kill browser.


[AVISO] PNG não salvo para confusion_matrix_best_model. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


Resorting to unclean kill browser.


...

Resorting to unclean kill browser.


[AVISO] SVG não salvo para classification_macro_f1_groupkfold. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


Modelo sklearn salvo: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\models\best_sklearn_groupkfold_model.joblib
Melhor modelo oficial: LogisticRegression_balanced
Seletor: SelectKBest(k=100)


,model,selector,n_samples,n_groups,n_features_requested,n_features_effective,cv,status,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,macro_precision_mean,macro_precision_std,macro_recall_mean,macro_recall_std
13,LogisticRegression_balanced,SelectKBest(k=100),6506,1802,520,100,GroupKFold(5) por song_id,ok,0.530602,0.031624,0.511994,0.036809,0.487545,0.035362,0.548181,0.029215,0.487605,0.032191,0.511994,0.036809


,label,precision,recall,f1-score,support
0,Q1_High_Valence_High_Arousal,0.775177,0.545539,0.640394,3217.000000
1,Q2_Low_Valence_High_Arousal,0.363963,0.497547,0.420398,1019.000000
2,Q3_Low_Valence_Low_Arousal,0.533592,0.597685,0.563823,1382.000000
3,Q4_High_Valence_Low_Arousal,0.279785,0.409910,0.332572,888.000000
4,accuracy,0.530587,0.530587,0.530587,0.530587
5,macro avg,0.488129,0.512670,0.489297,6506.000000
6,weighted avg,0.591837,0.530587,0.547658,6506.000000


## 6.2 Análise detalhada por fold — estabilidade e classe mais difícil

Esta seção expande a análise fold a fold do melhor modelo.
Para cada fold, identifica qual classe apresentou o pior F1-score.

> **Por que isso importa para o TCC?** A análise por fold mostra se o modelo é estável
> (performance similar em todos os folds) ou se depende de uma divisão específica das músicas.

In [11]:
# =========================
# TABELA FOLD A FOLD COM PIOR CLASSE
# =========================

if "best_fold_metrics" in dir() and "best_oof_pred" in dir():
    worst_class_per_fold = {}
    fold_per_class_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        y_test = y.iloc[test_idx]
        y_pred_fold = best_oof_pred.iloc[test_idx].astype(str)

        rep = classification_report(
            y_test, y_pred_fold,
            labels=labels_present,
            output_dict=True,
            zero_division=0,
        )
        class_f1 = {cls: rep.get(cls, {}).get("f1-score", 0.0) for cls in labels_present}
        worst = min(class_f1, key=class_f1.get)
        worst_class_per_fold[fold_idx] = short_quadrant_label(worst)

        for cls in labels_present:
            fold_per_class_rows.append({
                "fold": fold_idx,
                "class": short_quadrant_label(cls),
                "precision": rep.get(cls, {}).get("precision", 0.0),
                "recall": rep.get(cls, {}).get("recall", 0.0),
                "f1": rep.get(cls, {}).get("f1-score", 0.0),
                "support": rep.get(cls, {}).get("support", 0),
            })

    fold_detail = best_fold_metrics.copy()
    fold_detail["worst_class"] = fold_detail["fold"].map(worst_class_per_fold)
    fold_display = fold_detail[["fold", "accuracy", "balanced_accuracy", "macro_f1", "worst_class"]].copy()
    fold_display.columns = ["Fold", "Accuracy", "Balanced Acc", "Macro-F1", "Classe mais difícil"]
    fold_display = fold_display.round(4)

    display(Markdown(f"### Resultados por fold — {BEST_MODEL_NAME} / {BEST_SELECTOR_NAME}"))
    display(fold_display)
    save_table(fold_detail, "best_model_fold_detail_with_worst_class.csv")

    # Gráfico: F1 por classe e por fold
    fold_per_class_df = pd.DataFrame(fold_per_class_rows)
    if not fold_per_class_df.empty:
        fig = px.line(
            fold_per_class_df,
            x="fold",
            y="f1",
            color="class",
            markers=True,
            title=f"F1 por classe e por fold — {BEST_MODEL_NAME}",
        )
        fig.update_xaxes(dtick=1, title="Fold")
        fig.update_yaxes(range=[0, 1.05], title="F1-score")
        save_fig(fig, "tcc_09_f1_per_class_per_fold", legend_title="Classe")
        save_table(fold_per_class_df, "best_model_fold_per_class_metrics.csv")

    # Resumo de estabilidade
    f1_std = best_fold_metrics["macro_f1"].std(ddof=1) if len(best_fold_metrics) > 1 else 0
    stability = "estável (std < 0.03)" if f1_std < 0.03 else f"moderadamente variável (std = {f1_std:.4f})"
    worst_counts = pd.Series(worst_class_per_fold.values()).value_counts()
    worst_most = worst_counts.index[0] if len(worst_counts) > 0 else "N/A"
    display(Markdown(
        f"\n**Estabilidade:** Macro-F1 std = {f1_std:.4f} — modelo {stability}.\n\n"
        f"**Classe sistematicamente mais difícil:** {worst_most} "
        f"(pior em {worst_counts.iloc[0]} de {len(worst_class_per_fold)} folds)."
    ))
else:
    print("[AVISO] best_fold_metrics ou best_oof_pred não disponíveis. Execute a seção 6 primeiro.")


### Resultados por fold — LogisticRegression_balanced / SelectKBest(k=100)

,Fold,Accuracy,Balanced Acc,Macro-F1,Classe mais difícil
0,1,0.4820,0.4535,0.4289,Q4 Calmo/Relaxado
1,2,0.5665,0.5528,0.5221,Q4 Calmo/Relaxado
2,3,0.5488,0.5244,0.5061,Q4 Calmo/Relaxado
3,4,0.5273,0.5056,0.4881,Q4 Calmo/Relaxado
4,5,0.5285,0.5237,0.4925,Q4 Calmo/Relaxado


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\best_model_fold_detail_with_worst_class.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_09_f1_per_class_per_fold.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_09_f1_per_class_per_fold.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_09_f1_per_class_per_fold.svg


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\best_model_fold_per_class_metrics.csv



**Estabilidade:** Macro-F1 std = 0.0354 — modelo moderadamente variável (std = 0.0354).

**Classe sistematicamente mais difícil:** Q4 Calmo/Relaxado (pior em 5 de 5 folds).

In [12]:
# =========================
# COMPARAÇÃO: DUMMY vs openSMILE BASELINE
# Objetivo: provar que o modelo aprende além do acaso
# =========================

if "ok_results" in dir() and not ok_results.empty:
    dummy_rows = ok_results[ok_results["model"].str.startswith("Dummy")].copy()
    non_dummy_rows = ok_results[~ok_results["model"].str.startswith("Dummy")].copy()

    best_dummy_most = dummy_rows[dummy_rows["model"] == "Dummy_most_frequent"].sort_values("macro_f1_mean", ascending=False)
    best_dummy_strat = dummy_rows[dummy_rows["model"] == "Dummy_stratified"].sort_values("macro_f1_mean", ascending=False)
    best_non_dummy = non_dummy_rows.sort_values("macro_f1_mean", ascending=False).iloc[0] if not non_dummy_rows.empty else None

    ref_dummy_f1 = float(best_dummy_most.iloc[0]["macro_f1_mean"]) if not best_dummy_most.empty else 0.0

    comparison_rows = []
    for label, rows in [("Dummy most_frequent", best_dummy_most), ("Dummy stratified", best_dummy_strat)]:
        if rows.empty:
            continue
        row = rows.iloc[0]
        comparison_rows.append({
            "Modelo": label,
            "Macro-F1 médio": round(row.get("macro_f1_mean", float("nan")), 4),
            "Balanced Acc médio": round(row.get("balanced_accuracy_mean", float("nan")), 4),
            "Ganho vs Dummy most_frequent": "—",
            "Tipo": "Baseline trivial",
        })

    if best_non_dummy is not None:
        f1_best = float(best_non_dummy.get("macro_f1_mean", 0))
        bal_best = float(best_non_dummy.get("balanced_accuracy_mean", float("nan")))
        ganho = f1_best - ref_dummy_f1
        comparison_rows.append({
            "Modelo": f"{best_non_dummy['model']} [{best_non_dummy['selector']}]",
            "Macro-F1 médio": round(f1_best, 4),
            "Balanced Acc médio": round(bal_best, 4),
            "Ganho vs Dummy most_frequent": f"+{ganho:.4f}",
            "Tipo": "openSMILE baseline",
        })

    dummy_comparison_df = pd.DataFrame(comparison_rows)
    display(Markdown("### Comparação: Dummy vs openSMILE baseline"))
    display(dummy_comparison_df)
    save_table(dummy_comparison_df, "dummy_vs_opensmile_comparison.csv")

    if best_non_dummy is not None and ref_dummy_f1 > 0:
        ganho = float(best_non_dummy.get("macro_f1_mean", 0)) - ref_dummy_f1
        pct_ganho = (ganho / ref_dummy_f1) * 100 if ref_dummy_f1 > 0 else float("nan")
        display(Markdown(
            f"\n**Conclusão:** O melhor modelo openSMILE "
            f"({best_non_dummy['model']}) obteve Macro-F1 médio de "
            f"**{best_non_dummy['macro_f1_mean']:.4f}**, superando o melhor Dummy "
            f"em **+{ganho:.4f}** ({pct_ganho:.1f}% de ganho relativo). "
            f"Isso confirma que os descritores openSMILE capturam informação real sobre os quadrantes emocionais."
        ))
else:
    print("[AVISO] ok_results não disponível. Execute a seção 6 primeiro.")


### Comparação: Dummy vs openSMILE baseline

,Modelo,Macro-F1 médio,Balanced Acc médio,Ganho vs Dummy most_frequent,Tipo
0,Dummy most_frequent,0.1653,0.2500,—,Baseline trivial
1,Dummy stratified,0.2436,0.2448,—,Baseline trivial
2,LogisticRegression_balanced [SelectKBest(k=100)],0.4875,0.5120,+0.3223,openSMILE baseline


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\dummy_vs_opensmile_comparison.csv



**Conclusão:** O melhor modelo openSMILE (LogisticRegression_balanced) obteve Macro-F1 médio de **0.4875**, superando o melhor Dummy em **+0.3223** (195.0% de ganho relativo). Isso confirma que os descritores openSMILE capturam informação real sobre os quadrantes emocionais.

## 6.3 Comparação: GroupKFold vs StratifiedGroupKFold

O `GroupKFold` garante que músicas completas não apareçam em treino e teste simultaneamente
(evita vazamento), mas **não garante** distribuição equilibrada dos quadrantes em cada fold.
O `StratifiedGroupKFold` adiciona essa garantia.

Comparar os dois métodos permite avaliar:
- Se o modelo é sensível ao desbalanceamento de classes nos folds
- Se a metodologia `GroupKFold` é suficientemente robusta para o TCC

In [13]:
# =========================
# COMPARAÇÃO: GroupKFold vs StratifiedGroupKFold
# =========================

if StratifiedGroupKFold is not None and "best_pipeline" in dir():
    try:
        sgkf = StratifiedGroupKFold(n_splits=cv_splits)
        sgkf_fold_df, _ = evaluate_pipeline_groupkfold(
            clone(best_pipeline), X, y, groups, labels_present, sgkf
        )
        sgkf_summary = summarize_fold_metrics(sgkf_fold_df)

        sgkf_comparison = pd.DataFrame([
            {
                "Validação": f"GroupKFold({cv_splits})",
                "Macro-F1 médio": round(float(best_row.get("macro_f1_mean", float("nan"))), 4),
                "Macro-F1 std": round(float(best_row.get("macro_f1_std", float("nan"))), 4),
                "Balanced Acc médio": round(float(best_row.get("balanced_accuracy_mean", float("nan"))), 4),
                "Observação": "seguro por música",
            },
            {
                "Validação": f"StratifiedGroupKFold({cv_splits})",
                "Macro-F1 médio": round(sgkf_summary.get("macro_f1_mean", float("nan")), 4),
                "Macro-F1 std": round(sgkf_summary.get("macro_f1_std", float("nan")), 4),
                "Balanced Acc médio": round(sgkf_summary.get("balanced_accuracy_mean", float("nan")), 4),
                "Observação": "seguro por música + balanceia classes por fold",
            },
        ])

        display(Markdown("### GroupKFold vs StratifiedGroupKFold"))
        display(sgkf_comparison)
        save_table(sgkf_comparison, "groupkfold_vs_stratifiedgroupkfold.csv")
        save_table(sgkf_fold_df.assign(cv="StratifiedGroupKFold"), "stratifiedgroupkfold_fold_metrics.csv")

        delta = sgkf_summary["macro_f1_mean"] - float(best_row.get("macro_f1_mean", 0))
        if abs(delta) < 0.01:
            note = (
                f"Os resultados são muito próximos (|ΔF1| = {abs(delta):.4f} < 0.01). "
                "Isso indica que o modelo **não é sensível** à estratificação dos folds, "
                "e a metodologia `GroupKFold` é metodologicamente suficiente para o TCC."
            )
        else:
            note = (
                f"Há diferença relevante (ΔF1 = {delta:+.4f}) entre os dois protocolos. "
                "Isso indica sensibilidade ao balanceamento de classes por fold; "
                "considere reportar ambos os resultados no TCC."
            )
        display(Markdown(f"\n**Observação metodológica:** {note}"))

    except Exception as e:
        display(Markdown(f"**[AVISO]** StratifiedGroupKFold não executou: `{e}`"))
else:
    if StratifiedGroupKFold is None:
        display(Markdown(
            "**[AVISO]** `StratifiedGroupKFold` não disponível nesta versão do scikit-learn "
            "(disponível a partir da versão 1.0)."
        ))
    else:
        display(Markdown("**[AVISO]** `best_pipeline` não disponível. Execute a seção 6 primeiro."))


### GroupKFold vs StratifiedGroupKFold

,Validação,Macro-F1 médio,Macro-F1 std,Balanced Acc médio,Observação
0,GroupKFold(5),0.4875,0.0354,0.5120,seguro por música
1,StratifiedGroupKFold(5),0.4813,0.0306,0.5022,seguro por música + balanceia classes por fold


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\groupkfold_vs_stratifiedgroupkfold.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\stratifiedgroupkfold_fold_metrics.csv



**Observação metodológica:** Os resultados são muito próximos (|ΔF1| = 0.0063 < 0.01). Isso indica que o modelo **não é sensível** à estratificação dos folds, e a metodologia `GroupKFold` é metodologicamente suficiente para o TCC.

## 6.5 Comparação com split holdout 70/20/10 por song_id

Protocolo alternativo ao `GroupKFold`: divisão única das músicas em **treino (70%) / teste (20%) / validação (10%)**, agrupadas por `song_id` para manter a restrição anti-vazamento.

A divisão é feita sobre os `song_id` únicos — nenhuma música aparece em mais de um split. Os resultados são comparados com o `GroupKFold(5)` da seção 6 para verificar consistência.

> **Critério de consistência:** |ΔMacro-F1| < 0.05 indica que o modelo é robusto ao protocolo de avaliação. Uma diferença maior pode indicar instabilidade ou viés na divisão única.

In [14]:
# =========================
# SPLIT HOLDOUT 70 / 20 / 10 POR song_id
# Divisão por músicas — mantém restrição anti-vazamento
# Protocolo alternativo ao GroupKFold(5) conforme sugerido pelo orientador
# =========================

if "X" not in dir() or "model_df" not in dir():
    print("[AVISO] Células da seção 6 não executadas — pulando holdout.")
else:
    from sklearn.base import clone as _sk_clone

    # ── 1. Dividir músicas ────────────────────────────────────────────────
    _all_songs = np.array(sorted(model_df[SONG_ID_COL].unique()))
    _rng = np.random.default_rng(RANDOM_STATE)
    _rng.shuffle(_all_songs)

    _n_total  = len(_all_songs)
    _n_train  = int(0.70 * _n_total)
    _n_test   = int(0.20 * _n_total)
    # validação = restante (~10%)

    _songs_train = set(_all_songs[:_n_train])
    _songs_test  = set(_all_songs[_n_train : _n_train + _n_test])
    _songs_val   = set(_all_songs[_n_train + _n_test :])

    _mask_tr = model_df[SONG_ID_COL].isin(_songs_train)
    _mask_te = model_df[SONG_ID_COL].isin(_songs_test)
    _mask_va = model_df[SONG_ID_COL].isin(_songs_val)

    _Xtr = model_df.loc[_mask_tr, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _Xte = model_df.loc[_mask_te, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _Xva = model_df.loc[_mask_va, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _ytr = model_df.loc[_mask_tr, QUADRANT_COL].astype(str)
    _yte = model_df.loc[_mask_te, QUADRANT_COL].astype(str)
    _yva = model_df.loc[_mask_va, QUADRANT_COL].astype(str)

    print(f"Split holdout por song_id  (random_state={RANDOM_STATE})")
    print(f"  Músicas — treino: {len(_songs_train):>4} ({len(_songs_train)/_n_total:.1%})  "
          f"teste: {len(_songs_test):>4} ({len(_songs_test)/_n_total:.1%})  "
          f"val: {len(_songs_val):>4} ({len(_songs_val)/_n_total:.1%})")
    print(f"  Blocos  — treino: {_Xtr.shape[0]:>6}  teste: {_Xte.shape[0]:>6}  val: {_Xva.shape[0]:>6}")
    print()

    # ── 2. Treinar todos os candidatos ────────────────────────────────────
    _holdout_rows = []
    _sel_opts = [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]

    for _mname, _estimator in tqdm(build_candidate_models().items(), desc="Holdout"):
        for _k in _sel_opts:
            try:
                _pipe = build_pipeline(_sk_clone(_estimator), selector_k=_k, n_features=len(usable_feature_cols))
                _pipe.fit(_Xtr, _ytr)
                for _split, _Xe, _ye, _ns in [
                    ("test",       _Xte, _yte, len(_songs_test)),
                    ("validation", _Xva, _yva, len(_songs_val)),
                ]:
                    _ypred = _pipe.predict(_Xe)
                    _m = classification_metrics(_ye, _ypred)
                    _holdout_rows.append({
                        "model":    _mname,
                        "selector": f"SelectKBest(k={_k})" if _k else "none",
                        "split":    _split,
                        "n_songs":  _ns,
                        "n_blocks": len(_ye),
                        "cv":       "holdout_70_20_10",
                        "status":   "ok",
                        **_m,
                    })
            except Exception as _exc:
                _holdout_rows.append({
                    "model": _mname,
                    "selector": f"SelectKBest(k={_k})" if _k else "none",
                    "split": "test", "status": f"error: {_exc}",
                })

    holdout_df = pd.DataFrame(_holdout_rows)
    save_table(holdout_df, "classification_holdout_70_20_10_results.csv")

    # ── 3. Melhor modelo no conjunto de teste ─────────────────────────────
    _test_ok = holdout_df[(holdout_df["split"] == "test") & (holdout_df["status"] == "ok")]
    _best_h  = _test_ok.sort_values("macro_f1", ascending=False).iloc[0]

    display(Markdown("### Melhor modelo — Holdout teste (20% das músicas)"))
    display(pd.DataFrame([{
        "Modelo":    _best_h["model"],
        "Seletor":   _best_h["selector"],
        "Macro-F1":  round(_best_h["macro_f1"], 4),
        "Bal. Acc.": round(_best_h["balanced_accuracy"], 4),
        "N. blocos": int(_best_h["n_blocks"]),
        "N. músicas": int(_best_h["n_songs"]),
    }]))

    # ── 4. Comparação GroupKFold vs Holdout ───────────────────────────────
    if "best_row" in dir():
        _kf_f1 = float(best_row.get("macro_f1_mean", float("nan")))
        _ho_f1 = float(_best_h["macro_f1"])
        _delta  = _ho_f1 - _kf_f1

        _cmp = pd.DataFrame([
            {
                "Protocolo": f"GroupKFold(5) — {best_row.get('model', '?')} / {best_row.get('selector', '?')}",
                "Macro-F1":  round(_kf_f1, 4),
                "Bal. Acc.": round(float(best_row.get("balanced_accuracy_mean", float("nan"))), 4),
                "Nota": "média de 5 folds",
            },
            {
                "Protocolo": f"Holdout teste — {_best_h['model']} / {_best_h['selector']}",
                "Macro-F1":  round(_ho_f1, 4),
                "Bal. Acc.": round(float(_best_h["balanced_accuracy"]), 4),
                "Nota": f"20% das músicas ({_best_h['n_songs']} músicas)",
            },
        ])
        save_table(_cmp, "classification_kfold_vs_holdout_comparison.csv")
        display(Markdown("### Comparação: GroupKFold(5) vs Holdout 70/20/10"))
        display(_cmp)
        print(f"  Δ Macro-F1 (holdout − kfold) = {_delta:+.4f}")

    # ── 5. Relatório por classe e matriz de confusão — conjunto de teste ──
    _best_k_val = None if _best_h["selector"] == "none" \
                  else int(re.search(r"k=(\d+)", _best_h["selector"]).group(1))
    _best_pipe_h = build_pipeline(
        build_candidate_models()[_best_h["model"]],
        selector_k=_best_k_val,
        n_features=len(usable_feature_cols),
    )
    _best_pipe_h.fit(_Xtr, _ytr)
    _ypred_te = _best_pipe_h.predict(_Xte)

    from sklearn.metrics import classification_report as _clf_report, confusion_matrix as _cm_fn

    _report_h = _clf_report(_yte, _ypred_te, labels=labels_present, output_dict=True, zero_division=0)
    _report_h_df = pd.DataFrame(_report_h).T.reset_index().rename(columns={"index": "label"})
    save_table(_report_h_df, "classification_report_holdout_test.csv")
    display(Markdown("### Relatório por quadrante — Holdout teste"))
    display(_report_h_df.round(3))

    _cm_norm = pd.DataFrame(
        _cm_fn(_yte, _ypred_te, labels=labels_present, normalize="true"),
        index=labels_present, columns=labels_present,
    ).round(3)
    save_table(_cm_norm.reset_index().rename(columns={"index": "actual"}),
               "confusion_matrix_holdout_test_normalized.csv")

    _fig_cm_h = px.imshow(
        _cm_norm, text_auto=True, aspect="auto", color_continuous_scale="Blues",
        title=f"Matriz de confusão normalizada — Holdout teste | {_best_h['model']} / {_best_h['selector']}",
    )
    _fig_cm_h.update_layout(xaxis_title="Predito", yaxis_title="Real")
    save_fig(_fig_cm_h, "confusion_matrix_holdout_test_normalized")
    print("[Holdout 70/20/10] concluído.")

Split holdout por song_id  (random_state=42)
  Músicas — treino: 1261 (70.0%)  teste:  360 (20.0%)  val:  181 (10.0%)
  Blocos  — treino:   4547  teste:   1328  val:    631



Holdout: 100%|██████████| 9/9 [00:48<00:00,  5.44s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_holdout_70_20_10_results.csv


### Melhor modelo — Holdout teste (20% das músicas)

,Modelo,Seletor,Macro-F1,Bal. Acc.,N. blocos,N. músicas
0,RidgeClassifier_balanced,SelectKBest(k=60),0.4936,0.5074,1328,360


...

  Δ Macro-F1 (holdout − kfold) = +0.0061
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_report_holdout_test.csv


### Relatório por quadrante — Holdout teste

,label,precision,recall,f1-score,support
0,Q1_High_Valence_High_Arousal,0.722,0.589,0.649,630.00
1,Q2_Low_Valence_High_Arousal,0.457,0.491,0.474,228.00
2,Q3_Low_Valence_Low_Arousal,0.507,0.650,0.570,266.00
3,Q4_High_Valence_Low_Arousal,0.268,0.299,0.282,204.00
4,accuracy,0.540,0.540,0.540,0.54
5,macro avg,0.488,0.507,0.494,1328.00
6,weighted avg,0.564,0.540,0.547,1328.00


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\confusion_matrix_holdout_test_normalized.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\confusion_matrix_holdout_test_normalized.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\confusion_matrix_holdout_test_normalized.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\confusion_matrix_holdout_test_normalized.svg


[Holdout 70/20/10] concluído.


## 6.4 Comparação de cenários de seleção de features

Esta seção compara o melhor modelo com diferentes estratégias de seleção de features:

| Cenário | Descrição |
|---|---|
| Todas openSMILE | Usa todas as features aprovadas no filtro de qualidade |
| VarianceThreshold | Remove features com variância ≈ 0 |
| SelectKBest k=50 | As 50 features com maior ANOVA F-score |
| SelectKBest k=100 | As 100 features com maior ANOVA F-score |
| SelectKBest k=200 | As 200 features com maior ANOVA F-score |

> **Objetivo:** verificar se o openSMILE precisa de muitas features ou se poucas já explicam bem os quadrantes emocionais.

In [15]:
# =========================
# COMPARAÇÃO DE CENÁRIOS DE SELEÇÃO DE FEATURES
# Usa o melhor modelo da validação oficial com diferentes seletores
# =========================

if "best_pipeline" in dir() and "BEST_MODEL_NAME" in dir():
    best_estimator = build_candidate_models()[BEST_MODEL_NAME]

    scenarios = [("Todas openSMILE", None, None), ("VarianceThreshold", "variance", None)]
    for k in [50, 100, 200]:
        if k < len(usable_feature_cols):
            scenarios.append((f"SelectKBest k={k}", "kbest", k))

    feat_sel_rows = []
    for scenario_name, sel_type, sel_k in scenarios:
        try:
            if sel_type is None:
                pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("model", clone(best_estimator)),
                ])
                n_feat = len(usable_feature_cols)
            elif sel_type == "variance":
                from sklearn.feature_selection import VarianceThreshold as VT
                # mede quantas features sobrevivem
                X_imp = SimpleImputer(strategy="median").fit_transform(X)
                n_feat = int(VT(threshold=0.0).fit(X_imp).get_support().sum())
                pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("variance", VT(threshold=0.0)),
                    ("scaler", StandardScaler()),
                    ("model", clone(best_estimator)),
                ])
            else:
                k_eff = min(sel_k, len(usable_feature_cols))
                pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("selector", SelectKBest(score_func=f_classif, k=k_eff)),
                    ("scaler", StandardScaler()),
                    ("model", clone(best_estimator)),
                ])
                n_feat = k_eff

            fd, _ = evaluate_pipeline_groupkfold(pipe, X, y, groups, labels_present, cv)
            sm = summarize_fold_metrics(fd)
            feat_sel_rows.append({
                "Cenário": scenario_name,
                "Nº features": n_feat,
                "Modelo": BEST_MODEL_NAME,
                "Macro-F1 médio": round(sm["macro_f1_mean"], 4),
                "Macro-F1 std": round(sm.get("macro_f1_std", float("nan")), 4),
                "Balanced Acc médio": round(sm["balanced_accuracy_mean"], 4),
            })
        except Exception as e:
            feat_sel_rows.append({
                "Cenário": scenario_name,
                "Nº features": sel_k or len(usable_feature_cols),
                "Modelo": BEST_MODEL_NAME,
                "Macro-F1 médio": float("nan"),
                "Macro-F1 std": float("nan"),
                "Balanced Acc médio": float("nan"),
            })
            print(f"[AVISO] Erro em '{scenario_name}': {e}")

    feat_sel_df = pd.DataFrame(feat_sel_rows)
    display(Markdown(f"### Comparação de seleção de features — {BEST_MODEL_NAME}"))
    display(feat_sel_df)
    save_table(feat_sel_df, "feature_selection_comparison.csv")

    valid_rows = feat_sel_df.dropna(subset=["Macro-F1 médio"])
    if len(valid_rows) > 1:
        fig = px.bar(
            valid_rows.sort_values("Macro-F1 médio"),
            x="Macro-F1 médio",
            y="Cenário",
            orientation="h",
            error_x="Macro-F1 std",
            text=valid_rows.sort_values("Macro-F1 médio")["Macro-F1 médio"].map(lambda x: f"{x:.4f}"),
            title=f"Macro-F1 por cenário de seleção de features — {BEST_MODEL_NAME}",
        )
        fig.update_traces(textposition="outside")
        save_fig(fig, "tcc_10_feature_selection_comparison",
                 x_title="Macro-F1 médio", y_title="Cenário")

        best_sel = valid_rows.sort_values("Macro-F1 médio", ascending=False).iloc[0]
        all_f1 = valid_rows.sort_values("Macro-F1 médio")
        f1_range = all_f1["Macro-F1 médio"].max() - all_f1["Macro-F1 médio"].min()
        display(Markdown(
            f"\n**Melhor cenário:** {best_sel['Cenário']} "
            f"(Macro-F1 = **{best_sel['Macro-F1 médio']:.4f}**, {best_sel['Nº features']} features). "
            f"A variação entre cenários é de {f1_range:.4f} — "
            + ("pequena, indicando que poucas features já capturam bem os quadrantes."
               if f1_range < 0.02 else
               "relevante, indicando que a quantidade de features impacta o desempenho.")
        ))
else:
    print("[AVISO] best_pipeline não disponível. Execute a seção 6 primeiro.")


### Comparação de seleção de features — LogisticRegression_balanced

,Cenário,Nº features,Modelo,Macro-F1 médio,Macro-F1 std,Balanced Acc médio
0,Todas openSMILE,520,LogisticRegression_balanced,0.4677,0.0305,0.4838
1,VarianceThreshold,520,LogisticRegression_balanced,0.4677,0.0305,0.4838
2,SelectKBest k=50,50,LogisticRegression_balanced,0.4882,0.0353,0.5129
3,SelectKBest k=100,100,LogisticRegression_balanced,0.4875,0.0354,0.5120
4,SelectKBest k=200,200,LogisticRegression_balanced,0.4772,0.0306,0.4969


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\feature_selection_comparison.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_10_feature_selection_comparison.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_10_feature_selection_comparison.png
Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_10_feature_selection_comparison.svg



**Melhor cenário:** SelectKBest k=50 (Macro-F1 = **0.4882**, 50 features). A variação entre cenários é de 0.0205 — relevante, indicando que a quantidade de features impacta o desempenho.

In [16]:
# Hierarchical benchmark: arousal high/low -> valence high/low -> quadrant
# This uses the same best direct model family as a controlled comparison.


def build_hierarchical_binary_targets(df_in):
    out = df_in.copy()
    out["arousal_high"] = (pd.to_numeric(out[AROUSAL_COL], errors="coerce") >= AROUSAL_THRESHOLD).astype(int)
    out["valence_high"] = (pd.to_numeric(out[VALENCE_COL], errors="coerce") >= VALENCE_THRESHOLD).astype(int)
    return out



def combine_binary_predictions(valence_high_pred, arousal_high_pred, index=None):
    valence_high_pred = pd.Series(valence_high_pred).astype(int)
    arousal_high_pred = pd.Series(arousal_high_pred).astype(int)
    quadrant_pred = np.select(
        [
            (valence_high_pred.eq(1) & arousal_high_pred.eq(1)),
            (valence_high_pred.eq(0) & arousal_high_pred.eq(1)),
            (valence_high_pred.eq(0) & arousal_high_pred.eq(0)),
            (valence_high_pred.eq(1) & arousal_high_pred.eq(0)),
        ],
        QUADRANT_ORDER,
        default=None,
    )
    return pd.Series(quadrant_pred, index=index, dtype="object")



def evaluate_hierarchical_groupkfold(pipeline_factory, X, y_arousal, y_valence, y_quadrant, groups, cv):
    fold_rows = []
    oof_pred = pd.DataFrame(index=y_quadrant.index)

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y_quadrant, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_arousal_train, y_arousal_test = y_arousal.iloc[train_idx], y_arousal.iloc[test_idx]
        y_valence_train, y_valence_test = y_valence.iloc[train_idx], y_valence.iloc[test_idx]
        y_quadrant_test = y_quadrant.iloc[test_idx]

        arousal_model = clone(pipeline_factory())
        valence_model = clone(pipeline_factory())

        arousal_model.fit(X_train, y_arousal_train)
        valence_model.fit(X_train, y_valence_train)

        arousal_pred = pd.Series(arousal_model.predict(X_test), index=X_test.index)
        valence_pred = pd.Series(valence_model.predict(X_test), index=X_test.index)
        quadrant_pred = combine_binary_predictions(valence_pred, arousal_pred, index=X_test.index)

        oof_pred.loc[X_test.index, "pred_arousal_high"] = arousal_pred.values
        oof_pred.loc[X_test.index, "pred_valence_high"] = valence_pred.values
        oof_pred.loc[X_test.index, "prediction"] = quadrant_pred.values

        row = {
            "fold": fold_idx,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_groups_train": pd.Series(groups.iloc[train_idx]).nunique(),
            "n_groups_test": pd.Series(groups.iloc[test_idx]).nunique(),
        }
        row.update({f"arousal_{k}": v for k, v in classification_metrics(y_arousal_test, arousal_pred).items()})
        row.update({f"valence_{k}": v for k, v in classification_metrics(y_valence_test, valence_pred).items()})
        row.update({f"quadrant_{k}": v for k, v in classification_metrics(y_quadrant_test, quadrant_pred).items()})
        fold_rows.append(row)

    return pd.DataFrame(fold_rows), oof_pred



def summarize_prefixed_metrics(fold_df, prefix):
    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]
    summary = {}
    for metric in metric_cols:
        col = f"{prefix}_{metric}"
        summary[f"{col}_mean"] = fold_df[col].mean()
        summary[f"{col}_std"] = fold_df[col].std(ddof=1)
    return summary


hierarchical_targets = build_hierarchical_binary_targets(model_df)
y_arousal_bin = hierarchical_targets["arousal_high"].astype(int)
y_valence_bin = hierarchical_targets["valence_high"].astype(int)
y_quadrant_hier = hierarchical_targets[QUADRANT_COL].astype(str)

hierarchical_estimator_name = BEST_MODEL_NAME
hierarchical_selector_k = BEST_SELECTOR_K
hierarchical_selector_name = BEST_SELECTOR_NAME
hierarchical_pipeline_factory = lambda: build_pipeline(
    build_candidate_models()[hierarchical_estimator_name],
    selector_k=hierarchical_selector_k,
    n_features=len(usable_feature_cols),
)

hierarchical_fold_metrics, hierarchical_oof_pred = evaluate_hierarchical_groupkfold(
    hierarchical_pipeline_factory,
    X,
    y_arousal_bin,
    y_valence_bin,
    y_quadrant_hier,
    groups,
    cv,
)

hierarchical_results = pd.DataFrame([
    {
        "approach": "hierarchical_arousal_then_valence",
        "model": hierarchical_estimator_name,
        "selector": hierarchical_selector_name,
        "n_samples": len(model_df),
        "n_groups": n_groups,
        "n_features_requested": len(usable_feature_cols),
        "n_features_effective": len(usable_feature_cols) if hierarchical_selector_k is None else min(hierarchical_selector_k, len(usable_feature_cols)),
        "cv": f"GroupKFold({cv_splits}) por song_id",
        "status": "ok",
        **summarize_prefixed_metrics(hierarchical_fold_metrics, "arousal"),
        **summarize_prefixed_metrics(hierarchical_fold_metrics, "valence"),
        **summarize_prefixed_metrics(hierarchical_fold_metrics, "quadrant"),
    }
])

hierarchical_best_row = hierarchical_results.iloc[0]
save_table(hierarchical_results, "classification_hierarchical_groupkfold_results.csv")
save_table(hierarchical_fold_metrics, "classification_hierarchical_groupkfold_fold_metrics.csv")

hierarchical_oof_predictions = model_df[[SONG_ID_COL, VALENCE_COL, AROUSAL_COL, QUADRANT_COL]].copy()
if "block_id" in model_df.columns:
    hierarchical_oof_predictions["block_id"] = model_df["block_id"].values
if "block_start_sec" in model_df.columns:
    hierarchical_oof_predictions["block_start_sec"] = model_df["block_start_sec"].values
hierarchical_oof_predictions["pred_arousal_high"] = hierarchical_oof_pred["pred_arousal_high"].astype(int).values
hierarchical_oof_predictions["pred_valence_high"] = hierarchical_oof_pred["pred_valence_high"].astype(int).values
hierarchical_oof_predictions["prediction"] = hierarchical_oof_pred["prediction"].astype(str).values
hierarchical_oof_predictions["correct"] = hierarchical_oof_predictions[QUADRANT_COL].astype(str).eq(hierarchical_oof_predictions["prediction"].astype(str))
save_table(hierarchical_oof_predictions, "classification_hierarchical_oof_predictions.csv")

hierarchical_report = classification_report(
    y_quadrant_hier,
    hierarchical_oof_pred["prediction"].astype(str),
    labels=labels_present,
    output_dict=True,
    zero_division=0,
)
hierarchical_report_df = pd.DataFrame(hierarchical_report).T.reset_index().rename(columns={"index": "label"})
save_table(hierarchical_report_df, "classification_hierarchical_report_best_model.csv")

hierarchical_cm = confusion_matrix(y_quadrant_hier, hierarchical_oof_pred["prediction"].astype(str), labels=labels_present)
hierarchical_cm_df = pd.DataFrame(hierarchical_cm, index=labels_present, columns=labels_present)
hierarchical_cm_export = hierarchical_cm_df.reset_index().rename(columns={"index": "actual"})
save_table(hierarchical_cm_export, "confusion_matrix_hierarchical_best_model.csv")

fig_hierarchical_cm = px.imshow(
    hierarchical_cm_df,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title=f"Hierarchical OOF confusion matrix - {hierarchical_estimator_name} / {BEST_SELECTOR_NAME}",
)
fig_hierarchical_cm.update_layout(xaxis_title="Predicted", yaxis_title="Actual")
save_fig(fig_hierarchical_cm, "confusion_matrix_hierarchical_best_model")

hierarchical_stage_summary = pd.DataFrame([
    {
        "stage": "arousal",
        "accuracy_mean": hierarchical_best_row["arousal_accuracy_mean"],
        "balanced_accuracy_mean": hierarchical_best_row["arousal_balanced_accuracy_mean"],
        "macro_f1_mean": hierarchical_best_row["arousal_macro_f1_mean"],
    },
    {
        "stage": "valence",
        "accuracy_mean": hierarchical_best_row["valence_accuracy_mean"],
        "balanced_accuracy_mean": hierarchical_best_row["valence_balanced_accuracy_mean"],
        "macro_f1_mean": hierarchical_best_row["valence_macro_f1_mean"],
    },
    {
        "stage": "quadrant",
        "accuracy_mean": hierarchical_best_row["quadrant_accuracy_mean"],
        "balanced_accuracy_mean": hierarchical_best_row["quadrant_balanced_accuracy_mean"],
        "macro_f1_mean": hierarchical_best_row["quadrant_macro_f1_mean"],
    },
])
save_table(hierarchical_stage_summary, "classification_hierarchical_stage_summary.csv")

comparison_df = pd.DataFrame([
    {
        "approach": "direct_quadrant",
        "model": BEST_MODEL_NAME,
        "selector": BEST_SELECTOR_NAME,
        "macro_f1_mean": float(best_row["macro_f1_mean"]),
        "balanced_accuracy_mean": float(best_row["balanced_accuracy_mean"]),
        "weighted_f1_mean": float(best_row["weighted_f1_mean"]),
    },
    {
        "approach": "hierarchical_arousal_then_valence",
        "model": hierarchical_best_row["model"],
        "selector": hierarchical_best_row["selector"],
        "macro_f1_mean": float(hierarchical_best_row["quadrant_macro_f1_mean"]),
        "balanced_accuracy_mean": float(hierarchical_best_row["quadrant_balanced_accuracy_mean"]),
        "weighted_f1_mean": float(hierarchical_best_row["quadrant_weighted_f1_mean"]),
    },
])
save_table(comparison_df, "classification_direct_vs_hierarchical_comparison.csv")

fig_comparison = px.bar(
    comparison_df.melt(
        id_vars=["approach"],
        value_vars=["macro_f1_mean", "balanced_accuracy_mean"],
        var_name="metric",
        value_name="value",
    ),
    x="approach",
    y="value",
    color="metric",
    barmode="group",
    title="Direct vs hierarchical comparison",
)
fig_comparison.update_layout(xaxis_title="Approach", yaxis_title="Score")
save_fig(fig_comparison, "direct_vs_hierarchical_comparison")

print("Hierarchical benchmark completed:", hierarchical_best_row["quadrant_macro_f1_mean"])
display(hierarchical_results)
display(comparison_df)


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_hierarchical_groupkfold_results.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_hierarchical_groupkfold_fold_metrics.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_hierarchical_oof_predictions.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_hierarchical_report_best_model.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\confusion_matrix_hierarchical_best_model.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\confusion_matrix_hierarchical_best_model.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\f

Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_hierarchical_stage_summary.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\classification_direct_vs_hierarchical_comparison.csv
Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\direct_vs_hierarchical_comparison.html
Figura PNG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\direct_vs_hierarchical_comparison.png


Resorting to unclean kill browser.


Figura SVG salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\direct_vs_hierarchical_comparison.svg


Hierarchical benchmark completed: 0.4625798786121046


,approach,model,selector,n_samples,n_groups,n_features_requested,n_features_effective,cv,status,arousal_accuracy_mean,...,quadrant_balanced_accuracy_mean,quadrant_balanced_accuracy_std,quadrant_macro_f1_mean,quadrant_macro_f1_std,quadrant_weighted_f1_mean,quadrant_weighted_f1_std,quadrant_macro_precision_mean,quadrant_macro_precision_std,quadrant_macro_recall_mean,quadrant_macro_recall_std
0,hierarchical_arousal_then_valence,LogisticRegression_balanced,SelectKBest(k=100),6506,1802,520,100,GroupKFold(5) por song_id,ok,0.768678,...,0.477153,0.036069,0.46258,0.037717,0.548734,0.030605,0.46268,0.038986,0.477153,0.036069


,approach,model,selector,macro_f1_mean,balanced_accuracy_mean,weighted_f1_mean
0,direct_quadrant,LogisticRegression_balanced,SelectKBest(k=100),0.487545,0.511994,0.548181
1,hierarchical_arousal_then_valence,LogisticRegression_balanced,SelectKBest(k=100),0.462580,0.477153,0.548734


## 6.1 Gráficos finais de desempenho e erro

Esta seção complementa a validação oficial com figuras mais interpretáveis: comparação de modelos, estabilidade por fold, métricas por classe, matriz de confusão normalizada com percentuais e comparação entre abordagem direta e hierárquica.


In [17]:
# Gráficos finais de desempenho para relatório/TCC
ok_plot = ok_results.copy().sort_values("macro_f1_mean", ascending=False).head(20)
ok_plot["model_selector"] = ok_plot["model"] + " | " + ok_plot["selector"]
ok_plot = ok_plot.sort_values("macro_f1_mean")
fig = go.Figure()
fig.add_trace(go.Bar(
    x=ok_plot["macro_f1_mean"],
    y=ok_plot["model_selector"],
    orientation="h",
    name="Macro-F1",
    error_x=dict(type="data", array=ok_plot["macro_f1_std"].fillna(0)),
    hovertemplate="%{y}<br>Macro-F1=%{x:.4f}<extra></extra>",
))
fig.add_vline(x=ok_plot["macro_f1_mean"].max(), line_dash="dot", line_color="gray")
save_fig(fig, "tcc_06_top_models_macro_f1", x_title="Macro-F1 médio", y_title="Modelo e seletor", height=900)

# Estabilidade por fold do melhor modelo
best_fold_plot = best_fold_metrics.copy()
best_fold_plot["fold_label"] = best_fold_plot["fold"].map(lambda x: f"Fold {x}")
fold_long = best_fold_plot.melt(
    id_vars=["fold", "fold_label"],
    value_vars=["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"],
    var_name="metric",
    value_name="value",
)
metric_label_map = {"accuracy": "Accuracy", "balanced_accuracy": "Balanced Acc.", "macro_f1": "Macro-F1", "weighted_f1": "Weighted-F1"}
fold_long["metric"] = fold_long["metric"].map(metric_label_map)
fig = px.line(
    fold_long,
    x="fold_label",
    y="value",
    color="metric",
    markers=True,
    title=f"Estabilidade por fold — {BEST_MODEL_NAME} / {BEST_SELECTOR_NAME}",
)
fig.update_yaxes(range=[0, max(1.0, fold_long["value"].max() + 0.05)])
save_fig(fig, "tcc_07_best_model_fold_stability", x_title="Fold", y_title="Valor da métrica", legend_title="Métrica")

# Métricas por classe do melhor modelo
per_class = report_df[report_df["label"].isin(labels_present)].copy()
per_class["label_short"] = per_class["label"].map(short_quadrant_label)
per_class_long = per_class.melt(
    id_vars=["label", "label_short", "support"],
    value_vars=["precision", "recall", "f1-score"],
    var_name="metric",
    value_name="value",
)
per_class_long["metric"] = per_class_long["metric"].map({"precision": "Precisão", "recall": "Recall", "f1-score": "F1-score"})
fig = px.bar(
    per_class_long,
    x="label_short",
    y="value",
    color="metric",
    barmode="group",
    text=per_class_long["value"].map(lambda x: f"{x:.2f}"),
    title="Desempenho por quadrante emocional",
    hover_data={"support": True, "value": ":.4f"},
)
fig.update_traces(textposition="outside")
fig.update_yaxes(range=[0, min(1.05, per_class_long["value"].max() + 0.20)])
save_fig(fig, "tcc_08_per_class_metrics", x_title="Quadrante", y_title="Valor", legend_title="Métrica")

# Matriz de confusão normalizada com texto percentual
cm_percent = (cm_norm_df * 100).round(1)
fig = px.imshow(
    cm_percent,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Matriz de confusão normalizada por classe real (%)",
    labels=dict(x="Predito", y="Real", color="%"),
)
fig.update_xaxes(ticktext=[short_quadrant_label(x) for x in cm_percent.columns], tickvals=list(range(len(cm_percent.columns))))
fig.update_yaxes(ticktext=[short_quadrant_label(x) for x in cm_percent.index], tickvals=list(range(len(cm_percent.index))))
save_fig(fig, "tcc_09_confusion_matrix_normalized_percent", x_title="Predito", y_title="Real")

# Principais confusões fora da diagonal
cm_errors = cm_df.copy()
error_rows = []
for actual in cm_errors.index:
    row_sum = cm_errors.loc[actual].sum()
    for pred in cm_errors.columns:
        if actual == pred:
            continue
        n = int(cm_errors.loc[actual, pred])
        if n > 0:
            error_rows.append({
                "real": actual,
                "predito": pred,
                "real_short": short_quadrant_label(actual),
                "predito_short": short_quadrant_label(pred),
                "n": n,
                "percentual_na_classe_real": 100 * n / row_sum if row_sum else 0,
            })
error_df = pd.DataFrame(error_rows).sort_values("n", ascending=False)
save_table(error_df, "confusion_error_pairs_best_model.csv")
if not error_df.empty:
    err_plot = error_df.head(12).copy()
    err_plot["erro"] = err_plot["real_short"] + " → " + err_plot["predito_short"]
    err_plot = err_plot.sort_values("n")
    fig = px.bar(
        err_plot,
        x="n",
        y="erro",
        orientation="h",
        text="n",
        title="Principais confusões entre quadrantes",
        hover_data={"percentual_na_classe_real": ":.2f"},
    )
    fig.update_traces(textposition="outside")
    save_fig(fig, "tcc_10_top_confusion_pairs", x_title="Quantidade de erros", y_title="Erro")

# Comparação direta vs hierárquica, se existir
if "comparison_df" in globals() and not comparison_df.empty:
    comp_plot = comparison_df.copy()
    comp_long = comp_plot.melt(
        id_vars=["approach", "model", "selector"],
        value_vars=["balanced_accuracy_mean", "macro_f1_mean", "weighted_f1_mean"],
        var_name="metric",
        value_name="value",
    )
    comp_long["metric"] = comp_long["metric"].map({
        "balanced_accuracy_mean": "Balanced Acc.",
        "macro_f1_mean": "Macro-F1",
        "weighted_f1_mean": "Weighted-F1",
    })
    fig = px.bar(
        comp_long,
        x="approach",
        y="value",
        color="metric",
        barmode="group",
        text=comp_long["value"].map(lambda x: f"{x:.3f}"),
        title="Comparação entre classificação direta e abordagem hierárquica",
    )
    fig.update_traces(textposition="outside")
    save_fig(fig, "tcc_11_direct_vs_hierarchical", x_title="Abordagem", y_title="Valor da métrica", legend_title="Métrica")

save_plot_index()


Figura HTML salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\figures\tcc_06_top_models_macro_f1.html


Resorting to unclean kill browser.


[AVISO] PNG não salvo para tcc_06_top_models_macro_f1. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


Resorting to unclean kill browser.


[AVISO] SVG não salvo para tcc_06_top_models_macro_f1. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


...

[AVISO] PNG não salvo para tcc_11_direct_vs_hierarchical. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


Resorting to unclean kill browser.


[AVISO] SVG não salvo para tcc_11_direct_vs_hierarchical. Instale/atualize kaleido. Erro: Couldn't close or kill browser subprocess


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\figure_inventory.csv


'C:\\dev\\python\\TCC Ajustado\\Dados\\comparative_outputs\\opensmile_pycaret_outputs\\tables\\figure_inventory.csv'

## 7. Benchmark complementar com PyCaret

O PyCaret ? usado como benchmark secund?rio. O resultado oficial do notebook continua sendo a valida??o sklearn com `GroupKFold` por `song_id`.

In [18]:
pycaret_compare_df = pd.DataFrame()
pycaret_best_model = None
pycaret_final_model = None

if RUN_PYCARET:
    if not PYCARET_AVAILABLE:
        print("[AVISO] RUN_PYCARET=True, mas PyCaret n?o est? dispon?vel.")
        print(PYCARET_IMPORT_ERROR)
    else:
        pycaret_data = model_df[usable_feature_cols + [QUADRANT_COL, SONG_ID_COL]].copy()
        pycaret_data[QUADRANT_COL] = pycaret_data[QUADRANT_COL].astype(str)
        pycaret_groups = pycaret_data[SONG_ID_COL].astype(str)
        try:
            setup_kwargs = dict(
                data=pycaret_data,
                target=QUADRANT_COL,
                ignore_features=[SONG_ID_COL],
                session_id=PYCARET_SESSION_ID,
                fold_strategy=GroupKFold(n_splits=cv_splits),
                fold_groups=pycaret_groups,
                normalize=True,
                imputation_type="simple",
                fix_imbalance=PYCARET_FIX_IMBALANCE,
                verbose=False,
                html=False,
            )
            _ = setup(**setup_kwargs)

            sort_metric = "F1"
            try:
                add_metric(
                    id="macro_f1",
                    name="Macro F1",
                    score_func=lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro", zero_division=0),
                    greater_is_better=True,
                )
                sort_metric = "Macro F1"
            except Exception as metric_error:
                print("[AVISO] Não foi possível adicionar Macro F1 customizado; usando F1 padrão do PyCaret.")
                print(metric_error)

            pycaret_best_model = compare_models(sort=sort_metric, turbo=True)
            pycaret_compare_df = pull()
            pycaret_compare_df.insert(0, "sort_metric_used", sort_metric)
            save_table(pycaret_compare_df, "pycaret_compare_quadrants.csv")

            model_for_finalize = pycaret_best_model
            if RUN_TUNING:
                model_for_finalize = tune_model(pycaret_best_model, optimize=sort_metric)
                tuned_df = pull()
                save_table(tuned_df, "pycaret_tuned_best_quadrants.csv")

            pycaret_final_model = finalize_model(model_for_finalize)
            if SAVE_MODEL_ARTIFACTS:
                pycaret_model_base = os.path.join(MODELS_PATH, "best_pycaret_quadrants")
                save_model(pycaret_final_model, pycaret_model_base)
                print("Modelo PyCaret salvo:", pycaret_model_base)

            display(pycaret_compare_df)
        except Exception as e:
            print(f"[AVISO] PyCaret n?o concluiu o benchmark: {type(e).__name__}: {e}")
else:
    print("RUN_PYCARET=False; benchmark PyCaret ignorado.")


[AVISO] Não foi possível adicionar Macro F1 customizado; usando F1 padrão do PyCaret.
name 'add_metric' is not defined


                                    Model  Accuracy     AUC  Recall   Prec.  \
lightgbm  Light Gradient Boosting Machine    0.5933  0.7969  0.5933  0.5794   
gbc          Gradient Boosting Classifier    0.5694  0.0000  0.5694  0.5902   
xgboost         Extreme Gradient Boosting    0.5856  0.7915  0.5856  0.5706   
rf               Random Forest Classifier    0.5784  0.7904  0.5784  0.5717   
et                 Extra Trees Classifier    0.5834  0.7847  0.5834  0.5674   
lda          Linear Discriminant Analysis    0.5068  0.0000  0.5068  0.5529   
lr                    Logistic Regression    0.5077  0.0000  0.5077  0.5453   
ridge                    Ridge Classifier    0.4980  0.0000  0.4980  0.5472   
ada                  Ada Boost Classifier    0.4879  0.0000  0.4879  0.5518   
svm                   SVM - Linear Kernel    0.4851  0.0000  0.4851  0.5342   
dt               Decision Tree Classifier    0.4370  0.6058  0.4370  0.4740   
knn                K Neighbors Classifier    0.3590 

,sort_metric_used,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,F1,Light Gradient Boosting Machine,0.5933,0.7969,0.5933,0.5794,0.5807,0.3756,0.3785,7.254
gbc,F1,Gradient Boosting Classifier,0.5694,0.0000,0.5694,0.5902,0.5770,0.3702,0.3720,119.396
xgboost,F1,Extreme Gradient Boosting,0.5856,0.7915,0.5856,0.5706,0.5728,0.3630,0.3656,53.154
rf,F1,Random Forest Classifier,0.5784,0.7904,0.5784,0.5717,0.5690,0.3607,0.3634,6.944
et,F1,Extra Trees Classifier,0.5834,0.7847,0.5834,0.5674,0.5640,0.3524,0.3574,1.302
lda,F1,Linear Discriminant Analysis,0.5068,0.0000,0.5068,0.5529,0.5235,0.2960,0.2997,0.332
lr,F1,Logistic Regression,0.5077,0.0000,0.5077,0.5453,0.5220,0.2903,0.2929,16.990
ridge,F1,Ridge Classifier,0.4980,0.0000,0.4980,0.5472,0.5155,0.2862,0.2901,1.154
ada,F1,Ada Boost Classifier,0.4879,0.0000,0.4879,0.5518,0.5077,0.2823,0.2887,11.302
svm,F1,SVM - Linear Kernel,0.4851,0.0000,0.4851,0.5342,0.5021,0.2681,0.2720,2.914


## 9. Checagem crítica do notebook

Checklist metodológica adicionada ao final da execução para apontar riscos comuns: desbalanceamento, escolha da métrica, validação por música, dependência excessiva da acurácia e problemas de legibilidade dos gráficos.


In [19]:
# Checagem crítica automática
crit_rows = []

max_class_pct = class_balance["percent"].max()
min_class_pct = class_balance["percent"].min()
crit_rows.append({
    "item": "Desbalanceamento de classes",
    "status": "atenção" if max_class_pct > 40 or min_class_pct < 15 else "ok",
    "evidencia": f"classe majoritária={max_class_pct:.2f}%; classe minoritária={min_class_pct:.2f}%",
    "acao_recomendada": "Usar macro-F1, balanced accuracy e class_weight; não usar accuracy como métrica principal.",
})

crit_rows.append({
    "item": "Validação por música",
    "status": "ok" if "GroupKFold" in str(best_row.get("cv", "")) else "atenção",
    "evidencia": str(best_row.get("cv", "")),
    "acao_recomendada": "Manter GroupKFold/StratifiedGroupKFold por song_id para evitar vazamento entre blocos da mesma música.",
})

if "comparison_df" in globals() and not comparison_df.empty:
    direct = comparison_df[comparison_df["approach"].str.contains("Direta", case=False, na=False)]
    hierarchical = comparison_df[comparison_df["approach"].str.contains("Hier", case=False, na=False)]
    if not direct.empty and not hierarchical.empty:
        delta = float(direct["macro_f1_mean"].max() - hierarchical["macro_f1_mean"].max())
        crit_rows.append({
            "item": "Direto vs. hierárquico",
            "status": "ok" if delta >= 0 else "atenção",
            "evidencia": f"delta macro-F1 direto - hierárquico = {delta:.4f}",
            "acao_recomendada": "Se o direto vencer em macro-F1, usá-lo como resultado principal e deixar o hierárquico como experimento secundário.",
        })

crit_rows.append({
    "item": "Gráficos para relatório",
    "status": "ok",
    "evidencia": "Foram gerados HTML + PNG/SVG quando kaleido estiver disponível.",
    "acao_recomendada": "Usar as figuras com prefixo tcc_ no relatório quinzenal e na seção de resultados.",
})

critical_check_df = pd.DataFrame(crit_rows)
save_table(critical_check_df, "critical_notebook_checklist.csv")
display(critical_check_df)


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\critical_notebook_checklist.csv


,item,status,evidencia,acao_recomendada
0,Desbalanceamento de classes,atenção,classe majoritária=49.45%; classe minoritária=...,"Usar macro-F1, balanced accuracy e class_weigh..."
1,Validação por música,ok,GroupKFold(5) por song_id,Manter GroupKFold/StratifiedGroupKFold por son...
2,Gráficos para relatório,ok,Foram gerados HTML + PNG/SVG quando kaleido es...,Usar as figuras com prefixo tcc_ no relatório ...


In [20]:
# =========================
# AUDITORIA ANTI-VAZAMENTO (leakage_audit_opensmile.csv)
# Documenta todos os pontos de controle de vazamento para defesa na banca
# =========================

def check_not_in_features(col):
    return "OK" if col not in usable_feature_cols else "FALHA"

leakage_rows = [
    {
        "Item": "song_id removido das features",
        "Coluna": SONG_ID_COL,
        "Status": check_not_in_features(SONG_ID_COL),
        "Evidência": f"Presente em usable_feature_cols: {SONG_ID_COL in usable_feature_cols}",
    },
    {
        "Item": "frametime / time removido (não existe no parquet de blocos)",
        "Coluna": TIME_COL,
        "Status": check_not_in_features(TIME_COL),
        "Evidência": f"Presente em usable_feature_cols: {TIME_COL in usable_feature_cols}",
    },
    {
        "Item": "valence removido das features",
        "Coluna": VALENCE_COL,
        "Status": check_not_in_features(VALENCE_COL),
        "Evidência": f"Presente em usable_feature_cols: {VALENCE_COL in usable_feature_cols}",
    },
    {
        "Item": "arousal removido das features",
        "Coluna": AROUSAL_COL,
        "Status": check_not_in_features(AROUSAL_COL),
        "Evidência": f"Presente em usable_feature_cols: {AROUSAL_COL in usable_feature_cols}",
    },
    {
        "Item": "quadrant_label removido das features",
        "Coluna": QUADRANT_COL,
        "Status": check_not_in_features(QUADRANT_COL),
        "Evidência": f"Presente em usable_feature_cols: {QUADRANT_COL in usable_feature_cols}",
    },
    {
        "Item": "quadrant (código curto) removido das features",
        "Coluna": "quadrant",
        "Status": check_not_in_features("quadrant"),
        "Evidência": f"Presente em usable_feature_cols: {'quadrant' in usable_feature_cols}",
    },
    {
        "Item": "filename/method removidos das features",
        "Coluna": "filename, method",
        "Status": "OK" if not any(c in usable_feature_cols for c in ["filename", "method"]) else "FALHA",
        "Evidência": "Colunas de meta-dado; não são numéricas — excluídas automaticamente",
    },
    {
        "Item": "Features são apenas os_mean__* e os_std__*",
        "Coluna": FEATURE_PREFIX + "*",
        "Status": "OK" if all(c.startswith(FEATURE_PREFIX) for c in usable_feature_cols) else "FALHA",
        "Evidência": f"Todas as {len(usable_feature_cols)} features usáveis começam com '{FEATURE_PREFIX}'",
    },
    {
        "Item": "Validação agrupada por música (GroupKFold)",
        "Coluna": SONG_ID_COL,
        "Status": "OK",
        "Evidência": f"GroupKFold({cv_splits} folds) por song_id — músicas não se misturam entre treino/teste",
    },
    {
        "Item": "song_id usado apenas como grupo, nunca como feature",
        "Coluna": SONG_ID_COL,
        "Status": check_not_in_features(SONG_ID_COL),
        "Evidência": "groups = model_df[SONG_ID_COL] passado separadamente, não entra em X",
    },
    {
        "Item": "StandardScaler fitado apenas no treino de cada fold",
        "Coluna": "N/A",
        "Status": "OK",
        "Evidência": "StandardScaler está dentro do Pipeline sklearn — fit ocorre apenas em X_train de cada fold",
    },
    {
        "Item": "SelectKBest fitado apenas no treino de cada fold",
        "Coluna": "N/A",
        "Status": "OK",
        "Evidência": "SelectKBest está dentro do Pipeline — sem look-ahead no conjunto de teste",
    },
    {
        "Item": "SimpleImputer fitado apenas no treino de cada fold",
        "Coluna": "N/A",
        "Status": "OK",
        "Evidência": "SimpleImputer está dentro do Pipeline — sem vazamento de estatísticas de imputação",
    },
]

leakage_audit_df = pd.DataFrame(leakage_rows)
n_fail = (leakage_audit_df["Status"] == "FALHA").sum()

save_table(leakage_audit_df, "leakage_audit_opensmile.csv")
display(Markdown("### Auditoria anti-vazamento — openSMILE baseline"))
display(leakage_audit_df[["Item", "Status", "Evidência"]])

if n_fail == 0:
    display(Markdown(
        "**✅ Auditoria completa: todas as verificações passaram.** "
        "Não foram detectados pontos de vazamento entre treino e teste."
    ))
else:
    display(Markdown(
        f"**⚠️ {n_fail} falha(s) detectada(s).** Revisar os itens com status FALHA antes de publicar os resultados."
    ))


Tabela salva: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\leakage_audit_opensmile.csv


### Auditoria anti-vazamento — openSMILE baseline

,Item,Status,Evidência
0,song_id removido das features,OK,Presente em usable_feature_cols: False
1,frametime / time removido (não existe no parqu...,OK,Presente em usable_feature_cols: False
2,valence removido das features,OK,Presente em usable_feature_cols: False
3,arousal removido das features,OK,Presente em usable_feature_cols: False
4,quadrant_label removido das features,OK,Presente em usable_feature_cols: False
5,quadrant (código curto) removido das features,OK,Presente em usable_feature_cols: False
6,filename/method removidos das features,OK,Colunas de meta-dado; não são numéricas — excl...
7,Features são apenas os_mean__* e os_std__*,OK,Todas as 520 features usáveis começam com 'os_'
8,Validação agrupada por música (GroupKFold),OK,GroupKFold(5 folds) por song_id — músicas não ...
9,"song_id usado apenas como grupo, nunca como fe...",OK,groups = model_df[SONG_ID_COL] passado separad...


**✅ Auditoria completa: todas as verificações passaram.** Não foram detectados pontos de vazamento entre treino e teste.

## 8. Resumo para TCC

Gera um texto curto com limiares, distribui??o das classes, melhor modelo oficial e principais m?tricas. O arquivo final ? salvo em Markdown na pasta de sa?da.

In [21]:
def format_class_distribution(class_balance_df):
    parts = []
    for _, row in class_balance_df.iterrows():
        parts.append(f"{row[QUADRANT_COL]}: {int(row['n'])} ({row['percent']:.2f}%)")
    return "; ".join(parts)


def format_metric(value):
    if pd.isna(value):
        return "n/a"
    return f"{float(value):.4f}"


best_summary = best_row.to_dict()
class_distribution_text = format_class_distribution(class_balance)

# Dummy comparação para o texto
baseline_rows = ok_results[ok_results["model"].str.startswith("Dummy")].copy()
best_baseline_f1_text = "n/a"
if not baseline_rows.empty:
    best_baseline = baseline_rows.sort_values("macro_f1_mean", ascending=False).iloc[0]
    best_baseline_f1_text = format_metric(best_baseline["macro_f1_mean"])
    ganho_dummy = float(best_row.get("macro_f1_mean", 0)) - float(best_baseline.get("macro_f1_mean", 0))
    ganho_dummy_text = f"+{ganho_dummy:.4f}"
else:
    ganho_dummy_text = "n/a"

# Classe melhor e pior por F1 (usando report_df do melhor modelo OOF)
best_class_text = "N/A"
worst_class_text = "N/A"
if "report_df" in dir() and not report_df.empty:
    class_rows = report_df[report_df["label"].isin(labels_present)].copy()
    if not class_rows.empty:
        class_rows["f1"] = pd.to_numeric(class_rows["f1-score"], errors="coerce")
        best_class_row = class_rows.sort_values("f1", ascending=False).iloc[0]
        worst_class_row = class_rows.sort_values("f1", ascending=True).iloc[0]
        best_class_text = (
            f"{short_quadrant_label(best_class_row['label'])} (F1={best_class_row['f1']:.4f})"
        )
        worst_class_text = (
            f"{short_quadrant_label(worst_class_row['label'])} (F1={worst_class_row['f1']:.4f})"
        )

# Feature selection comparison
feat_sel_note = ""
if "feat_sel_df" in dir() and not feat_sel_df.empty:
    valid_fs = feat_sel_df.dropna(subset=["Macro-F1 médio"])
    if not valid_fs.empty:
        best_fs = valid_fs.sort_values("Macro-F1 médio", ascending=False).iloc[0]
        feat_sel_note = (
            f" O melhor cenário de seleção de features foi **{best_fs['Cenário']}** "
            f"com {best_fs['Nº features']} features e Macro-F1 = {best_fs['Macro-F1 médio']:.4f}."
        )

# StratifiedGroupKFold note
sgkf_note = ""
if "sgkf_comparison" in dir() and not sgkf_comparison.empty:
    gkf_row = sgkf_comparison[sgkf_comparison["Validação"].str.startswith("Group")].iloc[0]
    sgkf_row = sgkf_comparison[sgkf_comparison["Validação"].str.startswith("Strat")].iloc[0]
    delta_sgkf = float(sgkf_row["Macro-F1 médio"]) - float(gkf_row["Macro-F1 médio"])
    sgkf_note = (
        f" A comparação entre `GroupKFold` e `StratifiedGroupKFold` mostrou uma diferença de "
        f"{delta_sgkf:+.4f} em Macro-F1, "
        + ("indicando robustez metodológica." if abs(delta_sgkf) < 0.01 else
           "indicando sensibilidade ao balanceamento por fold.")
    )

summary_md = f"""### Síntese do baseline openSMILE — Classificação por quadrantes emocionais

A partir dos valores contínuos de `valence` e `arousal`, foram definidos quatro quadrantes emocionais
no modelo circumplexo, com limiar fixo em 0 (escala -1 a 1). A distribuicao observada foi:
{class_distribution_text}.

**Resultado principal:** O melhor modelo foi **{best_row.get('model', 'N/A')}**
com seletor **{best_row.get('selector', 'none')}**, validado com
GroupKFold({cv_splits} folds) por `song_id` (sem vazamento entre músicas).
- Macro-F1 médio: **{format_metric(best_row.get('macro_f1_mean'))}**
  (+/-{format_metric(best_row.get('macro_f1_std'))})
- Balanced Accuracy médio: **{format_metric(best_row.get('balanced_accuracy_mean'))}**

**Ganho sobre Dummy:** Macro-F1 do melhor Dummy = {best_baseline_f1_text};
ganho do openSMILE baseline = **{ganho_dummy_text}**.
Isso confirma que os descritores openSMILE capturam informação real sobre a emoção musical.

**Desempenho por classe:** Melhor classe = {best_class_text};
classe mais difícil = {worst_class_text}.
A dificuldade nas classes limítrofes (tipicamente Q2 e Q4) é esperada
dado que ocupam regiões vizinhas no espaço VA e apresentam menor suporte relativo.{feat_sel_note}{sgkf_note}

**Nota para o TCC:** Estes resultados representam o **baseline openSMILE**,
estabelecendo a referência de desempenho para a comparação com a abordagem de
fingerprinting emocional proposta neste trabalho.
Uma melhora significativa do fingerprinting sobre este baseline constituirá
a evidência empírica central da contribuição do TCC.
"""

display(Markdown(summary_md))

# Salvar o texto como arquivo para usar direto no TCC
summary_path = os.path.join(TABLES_PATH, "tcc_baseline_opensmile_summary.md")
with open(summary_path, "w", encoding="utf-8") as f_out:
    f_out.write(summary_md)
print("Resumo TCC salvo em:", summary_path)


### Síntese do baseline openSMILE — Classificação por quadrantes emocionais

A partir dos valores contínuos de `valence` e `arousal`, foram definidos quatro quadrantes emocionais
no modelo circumplexo, com limiar fixo em 0 (escala -1 a 1). A distribuicao observada foi:
Q1_High_Valence_High_Arousal: 3217 (49.45%); Q2_Low_Valence_High_Arousal: 1019 (15.66%); Q3_Low_Valence_Low_Arousal: 1382 (21.24%); Q4_High_Valence_Low_Arousal: 888 (13.65%).

**Resultado principal:** O melhor modelo foi **LogisticRegression_balanced**
com seletor **SelectKBest(k=100)**, validado com
GroupKFold(5 folds) por `song_id` (sem vazamento entre músicas).
- Macro-F1 médio: **0.4875**
  (+/-0.0354)
- Balanced Accuracy médio: **0.5120**

**Ganho sobre Dummy:** Macro-F1 do melhor Dummy = 0.2436;
ganho do openSMILE baseline = **+0.2440**.
Isso confirma que os descritores openSMILE capturam informação real sobre a emoção musical.

**Desempenho por classe:** Melhor classe = Q1 Alegre/Energético (F1=0.6404);
classe mais difícil = Q4 Calmo/Relaxado (F1=0.3326).
A dificuldade nas classes limítrofes (tipicamente Q2 e Q4) é esperada
dado que ocupam regiões vizinhas no espaço VA e apresentam menor suporte relativo. O melhor cenário de seleção de features foi **SelectKBest k=50** com 50 features e Macro-F1 = 0.4882. A comparação entre `GroupKFold` e `StratifiedGroupKFold` mostrou uma diferença de -0.0062 em Macro-F1, indicando robustez metodológica.

**Nota para o TCC:** Estes resultados representam o **baseline openSMILE**,
estabelecendo a referência de desempenho para a comparação com a abordagem de
fingerprinting emocional proposta neste trabalho.
Uma melhora significativa do fingerprinting sobre este baseline constituirá
a evidência empírica central da contribuição do TCC.


Resumo TCC salvo em: C:\dev\python\TCC Ajustado\Dados\comparative_outputs\opensmile_pycaret_outputs\tables\tcc_baseline_opensmile_summary.md
